# PYORPS-Workshop: Trassenplanung mit Python

**Vom Kostenraster zur Freileitung**

Wir planen Kabel- und Freileitungstrassen mit **PYORPS** (*Python for Optimal Routes in Power Systems*):
Aus Geodaten entsteht ein Kostenraster, darauf suchen wir kostenoptimale Routen.
Alles läuft als Python-Code in diesem Notebook; am Ende starten wir die grafische Oberfläche (GUI) lokal.

| Teil | Thema | Was ihr dabei lernt | Zeit (grobe Schätzung ohne Übung) |
|:---:|---|---|:---:|
| 1 | Installation und Prüfung | PYORPS und die GUI auf dem eigenen Rechner einrichten | 15–25 min |
| 2 | Grundidee | Raster, Graph, kürzeste Route, Nachbarschaften | 10 min |
| 3 | Kostenraster aus mehreren Layern | Geodaten laden, Kosten zuweisen, Layer überlagern, Annahmen variieren | 35–45 min |
| 4 | Dijkstra und Delta-Stepping | Algorithmen verstehen; 1→1, 1→N, M→N und paarweise rechnen | 45–60 min |
| 5 | Geländemodell (DGM) und 3D-Routing | Steigung als Kosten, 2D und 3D vergleichen | 40–50 min |
| 6 | Freileitung | Masten, Spannfelder, Abstände zum Boden mit `ConstrainedPathFinder` | 50–60 min |
| 7 | GUI lokal betreiben | Dieselbe Planung ohne Code im Browser | 20–30 min |
| 8 | Zusammenfassung | Welches Werkzeug wofür, weiterführende Quellen | 5 min |

Die Zeiten sind eine Schätzung für Einsteiger. Mit den Übungen wird es ein langer Tag; sinnvoll sind zwei Blöcke
(Teil 1 bis 4, dann Teil 5 bis 7). Abschnitte, die für den ersten Kontakt entbehrlich sind, tragen in der Überschrift
**„Vertiefung (optional)“**.

**So arbeitet ihr mit dem Notebook:** Zellen von oben nach unten ausführen (`Umschalt + Enter`).
Die meisten Teile enden mit einer kurzen **Übung**. Die Daten liegen im Ordner `data/`; nach dem ersten Lauf
braucht das Notebook kein Internet mehr.

**Empfohlen:** 8 GB Arbeitsspeicher; ein Durchlauf aller Zellen dauert je nach Rechner etwa 15 bis 30 Minuten (Schätzung).

*Quellen:* Die Kostenwerte und Teile der Erklärungen beruhen auf dem Tutorial zum OSMSES-Workshop 2026
(Ordner `tutorial/` im Repository, [github.com/marhofmann/pyorps](https://github.com/marhofmann/pyorps)).

---
# Teil 1: Installation

## 1.1 Voraussetzungen

- **Python 3.12 bis 3.15** (64 Bit). Prüfen mit `python --version`.
- **Windows, Linux oder macOS.** Fertige Programmpakete (*Wheels*) gibt es für alle drei; ihr braucht **keinen C++-Compiler**.
  macOS braucht Version 15 oder neuer (Apple Silicon).
- Ein Webbrowser (für die GUI) und eine Internetverbindung für die Installation.

## 1.2 Virtuelle Umgebung und Installation

Wir installieren in eine eigene Umgebung, damit nichts mit anderen Projekten kollidiert.
Die folgenden Befehle gebt ihr in einem **Terminal** ein (nicht in dieser Zelle).

**Windows (PowerShell):**
```powershell
py -3.13 -m venv .venv
.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
pip install "pyorps[gui,examples]"
```

**Linux / macOS:**
```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
pip install "pyorps[gui,examples]"
```

Statt `-3.13` geht jede installierte Version von 3.12 bis 3.15 (`py -0` listet die installierten Versionen auf).
Der Zusatz in eckigen Klammern (`[gui,examples]`, englisch *extras*) holt zusätzliche Pakete: die Oberfläche (Dash, Karte, Tabellen) und Jupyter (`notebook`).
Weitere Zusätze: `[gpu]` (NVIDIA-Grafikkarte, CuPy), `[graph]` (weitere Graph-Bibliotheken), `[simplify]` (Routenvereinfachung).

## 1.3 Workshop-Dateien holen und Jupyter starten

```bash
git clone https://github.com/marhofmann/pyorps.git
cd pyorps/workshop
jupyter notebook
```

Der Ordner `workshop/` liegt im Repository. Ohne Git geht auch der ZIP-Download: auf der GitHub-Seite
*Code → Download ZIP*, entpacken und im Ordner `workshop/` weiterarbeiten.

> **Wichtig:** Die Pfade im Notebook sind relativ (`data/...`). Das Notebook muss deshalb in seinem Ordner `workshop/`
> laufen. Die Prüfzelle unten meldet, ob der Ordner `data/` gefunden wird.

**Nur für Entwickler:** Wer den neuesten Stand aus GitHub braucht, installiert mit `pip install -e ".[dev,full]"` aus dem Repository;
dafür sind ein C++-Compiler und `python setup.py build_ext --inplace` nötig (siehe die Installationsseite der Dokumentation).

## 1.4 Prüfen, ob alles da ist

Die nächste Zelle meldet Python, Betriebssystem, die Version von PYORPS und ob die übersetzten Erweiterungen
(Dijkstra, Delta-Stepping) geladen werden. Sie zeigt auch, welche optionalen Teile vorhanden sind.

In [ ]:
import platform
from pathlib import Path

import pyorps

version = pyorps.__version__
ort = Path(pyorps.__file__).resolve().parent
print(f"Python  {platform.python_version()}  auf {platform.system()} ({platform.machine()})")
print(f"pyorps  {version}")
print(f"Ort     {ort}")

# Mindestversion: die Funktionen dieses Workshops gibt es ab 0.5.0
teile = []
for zahl in version.replace("rc", ".").replace("a", ".").replace("b", ".").split(".")[:3]:
    teile.append(int(zahl) if zahl.isdigit() else 0)
if tuple(teile) < (0, 5, 0):
    print("\n⚠ Version zu alt: bitte  pip install --upgrade \"pyorps[gui,examples]\"  ausführen.")
if "site-packages" not in str(ort):
    print("\nHinweis: pyorps kommt nicht aus site-packages, sondern aus einem Quellordner "
          "(Entwicklerinstallation).")
if not Path("data/profiles").is_dir():
    print(f"\n⚠ Arbeitsordner {Path.cwd()} enthält keinen Ordner data/: "
          "bitte das Notebook im Ordner workshop/ ausführen.")

# Übersetzte Erweiterungen (Dijkstra, Delta-Stepping)
try:
    from pyorps.utils import path_algorithms  # noqa: F401
    print("\nErweiterungen:  ok (Dijkstra und Delta-Stepping geladen)")
except ImportError as fehler:
    print("\nErweiterungen:  FEHLT –", fehler)

# Optionale Teile
import warnings

for name, zusatz in (("dash", "gui"), ("cupy", "gpu"), ("yaml", "gui")):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")      # CuPy meldet ohne CUDA-Pfad eine Warnung
            __import__(name)
        print(f"{name:<14}  ok")
    except ImportError:
        print(f"{name:<14}  nicht installiert (Zusatz [{zusatz}])")

Sieht die Ausgabe gut aus (Version ab 0.5.0, Erweiterungen `ok`), kann es losgehen.
`cupy` ist nur mit NVIDIA-Grafikkarte nötig; wir rechnen im Workshop auf dem Prozessor.

### Übung Teil 1

1. Welche Python-Version und welche PYORPS-Version habt ihr? Notiert beides – bei Fragen hilft das.
2. Führt im Terminal `python -m pyorps.gui --help` aus und lest, welche Optionen es gibt (Teil 7 braucht sie).

---
# Teil 2: Grundidee

**Warum?** Eine Trasse für Kabel oder Freileitung muss Wiesen, Wald, Straßen, Gewässer und Schutzgebiete kreuzen, und jedes
Gelände kostet etwas anderes. Von Hand findet man dafür selten die billigste Lösung. Ein Rechner kann es, wenn wir das
Problem in eine Form bringen, die er versteht: ein Raster mit Kosten.

PYORPS macht aus der Trassenplanung ein **Kürzeste-Wege-Problem auf einem Graphen**:

1. **Raster:** Jede Zelle trägt einen Kostenwert, zum Beispiel die Baukosten in EUR/m für diese Landnutzung.
2. **Graph:** Jede Zelle ist ein Knoten. Kanten verbinden Nachbarzellen; das Gewicht einer Kante ist *Kosten × Länge*.
3. **Suche:** Ein Algorithmus (Dijkstra oder Delta-Stepping, Teil 4) findet die Verbindung mit den kleinsten Gesamtkosten.
4. **Ergebnis:** Eine georeferenzierte Linie mit Länge, Kosten und exportierbarer Geometrie (GeoJSON, GeoPackage).

**Begriffe in diesem Notebook:** Die berechnete Linie heißt **Route**. **Trasse** sagen wir nur für die Planungsaufgabe
("wir planen eine Trasse"), **Leitung** nur für die Freileitung in Teil 6.

## Kostenwerte

| Wert | Bedeutung |
|---|---|
| 0 | kostenlos (zum Beispiel vorhandenes Leerrohr) |
| 1 bis 65534 | wachsende Kosten (typisch einige hundert EUR/m) |
| **65535** | **gesperrt**: dort darf die Route nicht verlaufen |

Die Werte liegen als `uint16` im Raster. **Niedriger Wert = billig**, gesperrt = unendlich teuer.

## Nachbarschaften

Die *Nachbarschaft* bestimmt, in wie viele Richtungen die Suche von einer Zelle aus gehen darf.

| Nachbarschaft | Richtungen | größte Längenüberschätzung | Beschreibung |
|:---:|:---:|:---:|---|
| `r0` | 4 | 41,4 % | nur Nord, Süd, Ost, West |
| `r1` | 8 | 8,2 % | zusätzlich die Diagonalen |
| **`r2`** | **16** | **2,7 %** | zusätzlich Springerzüge (Standard) |
| `r3` | 32 | 1,3 % | erweiterte Springerzüge |

Ein *Springerzug* geht wie der Springer im Schach: zwei Zellen in eine Richtung und eine zur Seite.
Die Längenüberschätzung sagt, um wie viel länger als die Luftlinie eine Route im ungünstigsten Winkel höchstens wird,
wenn sie nur diese Schritte benutzen darf (Rechnung für lange Strecken auf einem leeren Raster).
Mehr Richtungen ergeben glattere, genauere Routen, kosten aber Rechenzeit. Für die meisten Aufgaben ist `r2` richtig.
Wir setzen sie in Teil 3 selbst ein (`neighborhood_str="r2"`).

---
# Teil 3: Kostenraster aus mehreren überlappenden Layern

**Warum mehrere Layer?** Planerinnen und Planer suchen nicht die kürzeste, sondern die günstigste Route, und dabei reden viele Datenquellen mit:
Nutzung des Bodens, Schutzgebiete, vorhandene Straßen und eigene Planungsvorgaben. Legen wir jede Quelle als eigenen Layer auf ein Raster,
wird jede Annahme einzeln sichtbar und lässt sich einzeln ändern, ohne alles neu zu zeichnen.

Ein Kostenraster ist ein Gitter. Jede Zelle sagt, was ein Meter Kabel in dieser Zelle kostet (EUR/m).
Die Werte sind ganze Zahlen vom Typ `uint16`; der Wert **65535 heißt verboten**.
Wir bauen das Raster in zwei Schritten: Ein **Basis-Layer** deckt die ganze Fläche ab und gibt jeder Zelle einen Grundwert.
Darüber legen wir **modifizierende Layer**, die sich überlappen dürfen. Jeder Layer ändert nur die Zellen, die er bedeckt:
er **ersetzt** den Wert oder **multipliziert** ihn mit einem Faktor, auch mit einem Faktor unter 1.
Die Layer werden nacheinander angewendet. Gesperrte Zellen bleiben dabei gesperrt, und bei zwei Faktoren auf derselben Zelle multiplizieren sich beide.

Unser Untersuchungsgebiet ist ein Fenster von 5 × 5 km im Kinzigtal bei Biberach, dort mündet das Harmersbachtal ein: ein Talboden, Hänge mit Wald und der Rand des Schwarzwalds.

| Layer | Quelle | Wirkung |
|---|---|---|
| Tatsächliche Nutzung (ALKIS) | LGL-BW, WFS | Basis: Grundwert je Nutzungsart |
| Naturschutzgebiete | LUBW, WFS | ersetzen durch 65535 |
| Biotope (Biotopkartierung) | LUBW, WFS | ersetzen durch 65535 (unsere Annahme) |
| FFH-Gebiete (Natura 2000) | LUBW, WFS | Faktor 1,75 |
| Wasserschutzgebiete | LUBW, WFS | Faktor 1,5 |
| Überschwemmungsgebiete | LUBW, WFS | Faktor 1,3 |
| Straßenbündelung | Straßenflächen aus ALKIS, Puffer 10 m | Faktor 0,8 |
| Planungssperrfläche | eigenes Polygon | ersetzen durch 65535 |

*© LGL-BW (ALKIS) und LUBW (Schutzgebiete, Biotope, Überschwemmungsgebiete), Datenlizenz Deutschland – Namensnennung – Version 2.0 (www.govdata.de/dl-de/by-2-0); die Daten wurden zugeschnitten, gerastert und umklassifiziert.*

In [ ]:
# Bausteine für diesen Abschnitt
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from shapely.geometry import box, LineString, Point
import rasterio
from rasterio.transform import array_bounds, rowcol

from pyorps import (PathFinder, GeoRasterizer, CostAssumptions,
                    initialize_geo_dataset, WFSResponseParsingError,
                    full_window_buffer_m, route_metrics)


def de(x, nd=0):
    """Zahl im deutschen Format: Punkt als Tausendertrenner, Komma als Dezimalzeichen."""
    return f"{x:,.{nd}f}".replace(",", "§").replace(".", ",").replace("§", ".")


# Untersuchungsgebiet: 5 × 5 km im Kinzigtal bei Biberach (EPSG:25832, Koordinaten in Metern)
BBOX = (427000, 5354000, 432000, 5359000)   # (min Rechtswert, min Hochwert, max Rechtswert, max Hochwert)
CRS = "EPSG:25832"
AUFLOESUNG = 5                              # Zellgröße in Metern
VERBOTEN = 65535                            # uint16-Maximum: die Zelle ist gesperrt

# Layer-Cache und Kostenraster liegen unter data/
LAYER_DIR = Path("data/layers")
LAYER_DIR.mkdir(parents=True, exist_ok=True)
RASTER_A = "data/cost_raster_a.tiff"
RASTER_B = "data/cost_raster_b.tiff"

### Layer laden: erst der Cache, dann das Netz

Die Layer kommen von WFS-Servern (Web Feature Service) der Landesbehörden.
Die Funktion `lade_layer` schaut zuerst in den Cache `data/layers/`. Nur wenn die Datei fehlt, lädt sie über pyorps
(`initialize_geo_dataset` und `load_data`), schneidet auf das Fenster zu und speichert das Ergebnis.
Wir behalten nur die Spalte, die das Kostenmodell braucht; so bleiben die Dateien klein.

**Kein Netz nötig:** Die Cache-Dateien liegen schon im Repository. Deshalb läuft dieses Notebook ohne Internet, und die Download-Zweige von `lade_layer` laufen im Workshop nicht.
Wollt ihr einen Layer neu laden, löscht seine Datei in `data/layers/` und führt die Zelle noch einmal aus.
Das Fenster steckt im Dateinamen: Wer `BBOX` ändert, bekommt neue Dateien und lädt neu; alte Dateien eines anderen Fensters werden nie benutzt.

In [ ]:
def layer_datei(name, bbox=BBOX):
    """Cache-Datei eines Layers; das Fenster steht im Namen, damit nie ein Layer des falschen Fensters gelesen wird."""
    return LAYER_DIR / f"{name}_{'_'.join(str(int(v)) for v in bbox)}.gpkg"


def lade_layer(name, wfs, bbox, spalte=None):
    """Layer aus dem Cache lesen; fehlt er, per WFS laden, auf bbox zuschneiden und speichern."""
    datei = layer_datei(name, bbox)
    if datei.exists():
        return gpd.read_file(datei), "Cache"
    try:
        ds = initialize_geo_dataset(wfs, bbox=bbox)            # WFS-Anfrage vorbereiten
        ds.load_data()                                         # herunterladen
        gdf = ds.data.clip(box(*bbox), keep_geom_type=True)    # der Server liefert auch Objekte, die über den Rand ragen
        gdf = gdf[[spalte, "geometry"] if spalte else ["geometry"]]
    except WFSResponseParsingError:                            # kein Objekt im Fenster
        gdf = gpd.GeoDataFrame({spalte: []} if spalte else None, geometry=[], crs=CRS)
    gdf.to_file(datei, driver="GPKG")
    return gdf, "WFS"


lubw = "https://rips-gdi.lubw.baden-wuerttemberg.de/arcgis/services/wfs"
alkis_wfs = {"url": "https://owsproxy.lgl-bw.de/owsproxy/wfs/WFS_LGL-BW_ALKIS?version=2.0.0",
             "layer": "Tatsaechliche Nutzung"}


def lubw_wfs(layer):
    return {"url": f"{lubw}/{layer}/MapServer/WFSServer", "layer": layer}


QUELLEN = {   # Name im Cache: (WFS-Anfrage, Spalte, Behörde)
    "alkis":           (alkis_wfs, "unterart_name", "LGL-BW"),
    "naturschutz":     (lubw_wfs("Naturschutzgebiet"), None, "LUBW"),
    "biotop":          (lubw_wfs("Biotopkartierung"), None, "LUBW"),
    "ffh":             (lubw_wfs("FFH_Gebiet"), None, "LUBW"),
    "wasserschutz":    (lubw_wfs("Wasserschutzgebiet"), None, "LUBW"),
    "ueberschwemmung": (lubw_wfs("Ueberschwemmungsgebiet"), None, "LUBW"),
}

layer, zeilen = {}, []
for name, (wfs, spalte, behoerde) in QUELLEN.items():
    layer[name], herkunft = lade_layer(name, wfs, BBOX, spalte)
    anteil = layer[name].area.sum() / box(*BBOX).area * 100
    zeilen.append((name, len(layer[name]), round(anteil, 1), behoerde, herkunft))

print(pd.DataFrame(zeilen, columns=["Layer", "Objekte", "Fläche [%]", "Quelle", "geladen aus"]).to_string(index=False))
print("Layer ohne Objekt im Fenster:", [n for n, g in layer.items() if len(g) == 0] or "keine")

### Basis-Layer: Kosten je Nutzungsart

**Leere Layer.** Die Tabelle oben zeigt, was im Fenster wirklich liegt. Ein Layer ohne Objekt ist leer:
der Server meldet das als Fehler (`WFSResponseParsingError`), `lade_layer` fängt ihn ab und speichert einen leeren Layer.
Die Naturschutzgebiete laden wir trotzdem mit, damit die Regel "Naturschutz = gesperrt" im Modell steht, wenn ihr das Fenster wechselt.
In unseren Läufen hatten die Vogelschutz- und Landschaftsschutzgebiete aus dem OSMSES-Tutorial in diesem Fenster keine Objekte; wir laden sie deshalb nicht.
Als zweiten Sperr-Layer nehmen wir die kartierten **Biotope**: eine Annahme für den Workshop, keine Rechtsauskunft.

Der ALKIS-Layer hat in der Spalte `unterart_name` die Nutzungsart jeder Fläche.
Wir ordnen jeder Art einen Wert in EUR/m zu. Die Kostenwerte stammen aus dem Tutorial zum OSMSES-Workshop 2026
(Ordner `tutorial/` im Repository, https://github.com/marhofmann/pyorps); Arten, die es dort nicht gibt, haben wir ergänzt (Kommentar `# neu`).
Eine Art ohne Eintrag behandelt pyorps (mit einer Warnung) als verboten, also als 65535. Deshalb prüft die Zelle unten mit `assert`,
dass jede Art im Fenster einen Wert hat.

**Auflösung:** 5 m pro Zelle, also 1000 × 1000 = 1 Million Zellen. Das reicht für Wege und Gräben,
und die Suchen der späteren Abschnitte laufen darauf in kurzer Zeit (bei 2,5 m wären es 4 Millionen Zellen).
Eine Zelle bekommt den Wert einer Fläche nur, wenn ihr Mittelpunkt in der Fläche liegt. Schmale verbotene Flächen können deshalb Löcher haben
oder ganz verschwinden. pyorps warnt davor lang und ausführlich (`on_thin_features`); wir schalten die Warnung hier ab.
Für eine echte Genehmigungsplanung würden wir feiner rastern.

In [ ]:
KOSTEN_A = {   # Variante A "wirtschaftlich": EUR/m je Nutzungsart (Spalte unterart_name)
    # --- verboten: Bebauung, Flugplatz, Abbau ---
    "Wohnbaufläche": VERBOTEN,
    "Gebäude- und Freifläche Industrie und Gewerbe": VERBOTEN,
    "Gebäude- und Freifläche Land- und Forstwirtschaft": VERBOTEN,
    "Handel und Dienstleistung": VERBOTEN,
    "Friedhof": VERBOTEN,
    "Tagebau/Grube/Steinbruch": VERBOTEN,                                  # neu (wie Bergbau)
    "Flugverkehr": VERBOTEN,
    "Fläche gemischter Nutzung": VERBOTEN,                                 # neu
    "Gebäude- und Freifläche Versorgungsanlage": VERBOTEN,                 # neu
    "Gebäude- und Freifläche Entsorgungsanlage": VERBOTEN,                 # neu
    "Gebäude- und Freifläche Sport, Freizeit und Erholung": VERBOTEN,      # neu
    "Landwirtschaftliche Betriebsfläche": VERBOTEN,                        # neu
    "Lagerfläche": VERBOTEN,                                               # neu
    "Öffentliche Zwecke": VERBOTEN,                                        # neu
    "Historische Anlage": VERBOTEN,                                        # neu
    "Medien und Kommunikation": VERBOTEN,                                  # neu
    # --- begehbar, nach Kosten sortiert ---
    "Unland/Vegetationslose Fläche": 92,
    "Weg": 97,
    "Sportanlage": 107,                                                    # neu (wie Sport-, Freizeit- und Erholungsfläche)
    "Graben": 120,
    "Platz": 152,
    "Parkplatz": 152,                                                      # neu
    "Parken": 152,                                                         # neu
    "Bach": 155,                                                           # neu (wie Fließgewässer)
    "Fluss": 155,                                                          # neu (wie Fließgewässer)
    "Kanal": 155,                                                          # neu (wie Fließgewässer)
    "See": 155,                                                            # neu (wie Stehendes Gewässer)
    "Teich": 155,                                                          # neu (wie Stehendes Gewässer)
    "Straßenverkehr": 178,
    "Grünland": 200,
    "Grünanlage": 200,                                                     # neu
    "Brachland": 200,                                                      # neu
    "Ackerland": 250,
    "Garten": 300,
    "Rebfläche": 300,                                                      # neu (wie Garten; im Fenster nicht vorhanden)
    "Laubholz": 365,
    "Nadelholz": 365,
    "Laub- und Nadelholz": 365,
    "Gehölz": 365,                                                         # neu
    "Bahnverkehr": 415,
}

# Beweis: jede Nutzungsart im Fenster hat einen Kostenwert
arten = set(layer["alkis"]["unterart_name"])
ohne_kosten = sorted(arten - set(KOSTEN_A))
print("Nutzungsarten im Fenster:", len(arten), "| ohne Kostenwert:", ohne_kosten or "keine")
assert not ohne_kosten, "Kostenwert fehlt für: " + ", ".join(ohne_kosten)


def basis_raster(kosten, aufloesung=AUFLOESUNG):
    """Rastert den ALKIS-Layer mit einer Kostentabelle und gibt den GeoRasterizer zurück (Raster in .raster)."""
    gr = GeoRasterizer(layer_datei("alkis"), CostAssumptions({"unterart_name": kosten}))
    gr.rasterize(resolution_in_m=aufloesung,            # Zellgröße in Metern
                 bounding_box=box(*BBOX),               # Rasterrand = Fenster, nicht die Ausdehnung der Daten
                 on_thin_features="ignore")             # keine lange Warnung (siehe Text)
    return gr


def zaehle(raster):
    """Anzahl der verbotenen Zellen."""
    return int((raster == VERBOTEN).sum())


gr = basis_raster(KOSTEN_A)
basis = gr.raster.copy()          # Kopie: die Layer unten ändern gr.raster an Ort und Stelle
assert array_bounds(*basis.shape, gr.transform) == BBOX
print(f"Raster: {basis.shape[0]} × {basis.shape[1]} Zellen, {basis.dtype}, Zellgröße {gr.transform.a:g} m")
print(f"verbotene Zellen: {de(zaehle(basis))} ({de(zaehle(basis) / basis.size * 100, 1)} %)")

# Darstellung: Farbe = EUR/m, schwarz = verboten
EXTENT = (BBOX[0], BBOX[2], BBOX[1], BBOX[3])
KARTE = plt.get_cmap("YlOrRd").copy()
KARTE.set_bad("black")                                  # maskierte Zellen erscheinen schwarz


def beschrifte(ax):
    """Achsen einer Karte: Koordinaten in Metern, Schrift mindestens 8 pt (für den Beamer)."""
    ax.set_xticks(range(BBOX[0], BBOX[2], 2000))
    ax.set_yticks(range(BBOX[1], BBOX[3], 2000))
    ax.ticklabel_format(useOffset=False, style="plain")
    ax.tick_params(labelsize=8)
    ax.set_xlabel("Rechtswert [m]", fontsize=9)
    ax.set_ylabel("Hochwert [m]", fontsize=9)


def zeige(ax, raster, titel, vmax=800, alpha=1.0):
    """Kostenraster zeichnen; Werte über vmax bekommen die dunkelste Farbe."""
    bild = ax.imshow(np.ma.masked_equal(raster, VERBOTEN), cmap=KARTE, vmin=0, vmax=vmax,
                     extent=EXTENT, origin="upper", interpolation="nearest", alpha=alpha)
    ax.set_title(titel, fontsize=10)
    beschrifte(ax)
    return bild


def farbleiste(fig, bild, ax):
    """Farbleiste mit Beschriftung."""
    leiste = fig.colorbar(bild, ax=ax, label="Kosten [EUR/m]", extend="max", shrink=0.8)
    leiste.ax.tick_params(labelsize=8)
    leiste.set_label("Kosten [EUR/m]", fontsize=9)
    return leiste


fig, ax = plt.subplots(figsize=(7, 6))
bild = zeige(ax, basis, "Basis-Kostenraster (Variante A, nur ALKIS)")
farbleiste(fig, bild, ax)
ax.legend(handles=[Patch(color="black", label="verboten (65535)")], loc="lower left", fontsize=8)
plt.show()

### Modifizierende Layer: Schutzgebiete und Überschwemmungsgebiete

Jetzt kommen die Layer von der LUBW obendrauf. Wir beschreiben jeden Layer mit zwei Angaben:
dem Wert und `multiply`. Mit `multiply=False` ersetzt der Wert die Zellen, mit `multiply=True` wird multipliziert.
`modify_raster_from_dataset` liest den Layer aus unserer GPKG-Datei, rastert ihn auf dasselbe Gitter und ändert `gr.raster` an Ort und Stelle.
Zellen mit dem Wert 65535 lässt die Methode standardmäßig unverändert (`ignore_value=65535`).
Leere Layer überspringen wir.

In [ ]:
SCHUTZ_A = {   # Layer im Cache: (Wert, multiply)
    "naturschutz":     (VERBOTEN, False),   # ersetzen: gesperrt
    "biotop":          (VERBOTEN, False),   # ersetzen: gesperrt (unsere Annahme)
    "ffh":             (1.75, True),        # Faktor
    "wasserschutz":    (1.5, True),         # Faktor
    "ueberschwemmung": (1.3, True),         # Faktor
}


def schutz_anwenden(gr, wirkung, ausgabe=True):
    """Wendet die Layer der LUBW nacheinander an und zählt die verbotenen Zellen davor und danach."""
    for name, (wert, multiplizieren) in wirkung.items():
        if len(layer[name]) == 0:
            if ausgabe:
                print(f"  {name:16s} übersprungen (kein Objekt im Fenster)")
            continue
        vorher = zaehle(gr.raster)
        gr.modify_raster_from_dataset(layer_datei(name), cost_assumptions=wert, multiply=multiplizieren)
        if ausgabe:
            art = f"Faktor {str(wert).replace('.', ',')}" if multiplizieren else f"ersetzen durch {wert}"
            print(f"  {name:16s} {art:22s} verbotene Zellen: {de(vorher):>8} -> {de(zaehle(gr.raster)):>8}")


print(f"nach dem Basis-Layer: {de(zaehle(gr.raster))} verbotene Zellen")
schutz_anwenden(gr, SCHUTZ_A)
mit_schutz = gr.raster.copy()

### Layer anderer Art: Puffer und eigene Polygone

Nicht jeder Layer kommt von einem WFS. `modify_raster_from_dataset` nimmt auch ein `GeoDataFrame` entgegen.
Wir zeigen zwei Arten:

1. **Straßenbündelung:** Wir wählen die Straßenflächen aus dem ALKIS-Layer, legen einen **Puffer von 10 m** darum (`geometry_buffer_m=10`)
   und multiplizieren mit **0,8**. Ein Faktor unter 1 macht die Zellen billiger: Wer neben einer Straße baut, spart Aufwand.
2. **Planungssperrfläche:** Ein Polygon, das wir selbst im Notebook zeichnen: ein geplantes Gewerbegebiet von 350 m × 300 m
   auf dem Talboden, das frei bleiben soll. Es ersetzt die Zellen durch den Wert 65535.

Unten sehen wir vier Bilder: Basis, mit den Layern der LUBW, mit Straßenbündelung und Sperrfläche und eine Karte der Änderungen.

In [ ]:
strassen = layer["alkis"][layer["alkis"]["unterart_name"] == "Straßenverkehr"]

# eigenes Polygon: ein GeoDataFrame mit einer Fläche; (x, y) ist die untere linke Ecke, 350 m breit, 300 m hoch
SPERR_X, SPERR_Y = 428100, 5357150
sperrflaeche = gpd.GeoDataFrame({"name": ["geplantes Gewerbegebiet"]},
                                geometry=[box(SPERR_X, SPERR_Y, SPERR_X + 350, SPERR_Y + 300)], crs=CRS)


def eigene_layer_anwenden(gr, buendelung):
    """Straßenbündelung (Puffer + Faktor) und Planungssperrfläche (ersetzen)."""
    gr.modify_raster_from_dataset(strassen, cost_assumptions=buendelung, multiply=True,
                                  geometry_buffer_m=10)          # Puffer in Metern, vor dem Rastern
    gr.modify_raster_from_dataset(sperrflaeche, cost_assumptions=VERBOTEN)   # multiply=False: ersetzen


vorher = zaehle(gr.raster)
eigene_layer_anwenden(gr, 0.8)
endraster_a = gr.raster.copy()
print(f"Straßenbündelung und Sperrfläche: verbotene Zellen {de(vorher)} -> {de(zaehle(endraster_a))}")
print(f"Basis -> Ende: {de(zaehle(basis))} -> {de(zaehle(endraster_a))} verbotene Zellen")

# Karte der Änderungen: Basis gegen Endraster
aenderung = np.zeros(basis.shape, dtype=np.uint8)
aenderung[(endraster_a > basis) & (endraster_a != VERBOTEN)] = 1      # teurer
aenderung[endraster_a < basis] = 2                                    # billiger
aenderung[(endraster_a == VERBOTEN) & (basis != VERBOTEN)] = 3        # neu gesperrt

fig, axes = plt.subplots(2, 2, figsize=(9, 6), constrained_layout=True)
for ax, raster_i, titel in zip(axes.ravel()[:3], (basis, mit_schutz, endraster_a),
                               ("1. Basis (ALKIS)", "2. mit Layern der LUBW", "3. mit Straßenbündelung\nund Sperrfläche")):
    bild = zeige(ax, raster_i, titel)
leiste = fig.colorbar(bild, ax=axes.ravel().tolist(), extend="max", shrink=0.6)
leiste.set_label("Kosten [EUR/m]", fontsize=9)
leiste.ax.tick_params(labelsize=8)
farben = ["#d9d9d9", "#f4a582", "#4393c3", "black"]
axes[1, 1].imshow(aenderung, cmap=ListedColormap(farben), vmin=0, vmax=3, extent=EXTENT,
                  origin="upper", interpolation="nearest")
axes[1, 1].set_title("4. Änderung gegenüber der Basis", fontsize=10)
beschrifte(axes[1, 1])
axes[1, 1].legend(handles=[Patch(color=f, label=t) for f, t in
                           zip(farben, ("unverändert", "teurer", "billiger", "neu gesperrt"))],
                  loc="lower left", fontsize=8)
plt.show()

### Zwei Kostenvarianten auf denselben Layern

Dieselben Layer, andere Zahlen: Variante A ist die **wirtschaftliche** (die Werte von oben).
Variante B ist **naturverträglich**: Wald und Schutzgebiete sind viel teurer, Gewässer sind teuer zu queren, Felder sind billiger.

| Parameter | A "wirtschaftlich" | B "naturverträglich" |
|---|---|---|
| Wald (Laubholz, Nadelholz, Gehölz) | 365 EUR/m | 657 EUR/m (Faktor 1,8) |
| Ackerland / Grünland | 250 / 200 EUR/m | 150 / 130 EUR/m |
| Bach, Fluss, Kanal, See, Teich | 155 EUR/m | 600 EUR/m |
| FFH- / Wasserschutz- / Überschwemmungsgebiet | Faktor 1,75 / 1,5 / 1,3 | Faktor 4 / 3 / 2 |
| Straßenbündelung | Faktor 0,8 | Faktor 0,6 |
| verboten (Bebauung, Biotope, Sperrfläche) | gleich | gleich |

Alle Schritte stecken in der Funktion `baue_raster`. Sie speichert das Ergebnis mit `save_raster` als GeoTIFF.
Variante A muss dasselbe Raster liefern wie die Einzelschritte oben; die Zelle prüft das mit `assert` und meldet es.

In [ ]:
KOSTEN_B = dict(KOSTEN_A)                                # Kopie, dann ändern
for wald in ["Laubholz", "Nadelholz", "Laub- und Nadelholz", "Gehölz"]:
    KOSTEN_B[wald] = round(KOSTEN_A[wald] * 1.8)         # Wald × 1,8
KOSTEN_B.update({"Ackerland": 150, "Grünland": 130,
                 "Bach": 600, "Fluss": 600, "Kanal": 600, "See": 600, "Teich": 600})
SCHUTZ_B = {**SCHUTZ_A, "ffh": (4.0, True), "wasserschutz": (3.0, True),
            "ueberschwemmung": (2.0, True)}


def baue_raster(kosten, schutz, buendelung, pfad):
    """Basis, Layer der LUBW, Straßenbündelung, Sperrfläche; speichert das Raster als GeoTIFF."""
    gr = basis_raster(kosten)
    schutz_anwenden(gr, schutz, ausgabe=False)
    eigene_layer_anwenden(gr, buendelung)
    gr.save_raster(pfad)
    return gr


VARIANTEN = {"A": dict(kosten=KOSTEN_A, schutz=SCHUTZ_A, buendelung=0.8, pfad=RASTER_A),
             "B": dict(kosten=KOSTEN_B, schutz=SCHUTZ_B, buendelung=0.6, pfad=RASTER_B)}
raster = {}
for v, parameter in VARIANTEN.items():
    raster[v] = baue_raster(**parameter).raster.copy()
    groesse = Path(parameter["pfad"]).stat().st_size / 1e6
    print(f"Variante {v}: {parameter['pfad']} ({de(groesse, 1)} MB), verbotene Zellen {de(zaehle(raster[v]))} "
          f"({de(zaehle(raster[v]) / raster[v].size * 100, 1)} %), mittlerer Wert der freien Zellen "
          f"{de(raster[v][raster[v] != VERBOTEN].mean())} EUR/m")

assert np.array_equal(raster["A"], endraster_a), "Funktion und Einzelschritte liefern verschiedene Raster"
with rasterio.open(RASTER_A) as src:                     # Gegenprobe: die Datei enthält dasselbe Raster
    assert np.array_equal(src.read(1), raster["A"])
    print(f"Datei: {src.width} × {src.height} Zellen, {src.dtypes[0]}, {src.crs}, Zellgröße {src.res[0]:g} m")
print("Funktion und Einzelschritte liefern dasselbe Raster A; die Datei enthält dasselbe Raster.")

### Dieselbe Strecke, zwei Kostenwelten

Jetzt rechnen wir mit `PathFinder` die günstigste Route von `SOURCE` nach `TARGET` (3 bis 4 km Luftlinie), einmal pro Raster.
`SOURCE` liegt auf dem Talboden, `TARGET` auf der anderen Seite eines bewaldeten Hangs.
Wir nutzen dabei das ganze Raster als Suchfenster (`search_space_buffer_m=full_window_buffer_m(...)`) und die 16er-Nachbarschaft `"r2"`.
Alle Punkte müssen auf freien Zellen liegen; sonst verschiebt pyorps sie mit einer Warnung auf eine andere Zelle. Das prüfen wir mit `assert`.
Die erste Tabelle zeigt auch, wie viele Meter jede Route durch Wald, durch Schutzgebiete (FFH und Wasserschutz) und durch Überschwemmungsgebiet führt.

**Vorsicht beim Vergleichen:** Die Kosten der Routen stammen aus zwei verschiedenen Kostentabellen, sie sind nicht direkt vergleichbar.
Fair ist die Gegenprobe: Wir bewerten jede Route auf beiden Rastern (`route_metrics` rechnet wie `PathFinder`;
die Diagonale der zweiten Tabelle stimmt mit den Kosten der ersten Tabelle überein).

Die Punktlisten `SOURCES` (3 Punkte) und `TARGETS` (4 Punkte) brauchen wir in den nächsten Abschnitten für Mehrfach-Suchen.

In [ ]:
SOURCE = (427558, 5357403)        # Rechtswert, Hochwert (EPSG:25832): auf dem Talboden der Kinzig
TARGET = (430288, 5355723)        # am Rand des Harmersbachtals, hinter einem bewaldeten Hang
SOURCES = [SOURCE, (428238, 5358313), (428663, 5356873)]
TARGETS = [TARGET, (431613, 5355083), (429853, 5355148), (431273, 5356048)]

# Alle Punkte müssen in beiden Rastern auf freien Zellen liegen
with rasterio.open(RASTER_A) as src:
    transform = src.transform
for v in raster:
    for punkt in [SOURCE, TARGET, *SOURCES, *TARGETS]:
        zeile, spalte = rowcol(transform, *punkt)
        assert raster[v][zeile, spalte] != VERBOTEN, f"{punkt} ist in Variante {v} gesperrt"

luftlinie = np.hypot(TARGET[0] - SOURCE[0], TARGET[1] - SOURCE[1])
assert 3000 <= luftlinie <= 4000, "SOURCE und TARGET sollen 3 bis 4 km auseinander liegen"

routen = {}
for v, pfad in (("A", RASTER_A), ("B", RASTER_B)):
    finder = PathFinder(dataset_source=pfad, source_coords=SOURCE, target_coords=TARGET,
                        search_space_buffer_m=full_window_buffer_m(pfad),   # ganzes Raster
                        neighborhood_str="r2")
    routen[v] = finder.find_route()                      # Dijkstra, eine Route

wald = layer["alkis"][layer["alkis"]["unterart_name"].isin(
    ["Laubholz", "Nadelholz", "Laub- und Nadelholz", "Gehölz"])].union_all()
schutzflaeche = pd.concat([layer["ffh"], layer["wasserschutz"]]).union_all()
ueberschwemmt = layer["ueberschwemmung"].union_all()

zeilen = []
for v, p in routen.items():
    g = p.path_geometry
    zeilen.append((v, round(p.total_length), round(p.total_cost), round(p.total_length / luftlinie, 2),
                   round(g.intersection(wald).length), round(g.intersection(schutzflaeche).length),
                   round(g.intersection(ueberschwemmt).length)))
print(pd.DataFrame(zeilen, columns=["Variante", "Länge [m]", "Kosten [EUR]", "Umweg-Faktor", "davon Wald [m]",
                                    "davon Schutzgebiet [m]", "davon Überschwemmungsgebiet [m]"]).to_string(index=False))
abstand = routen["A"].path_geometry.hausdorff_distance(routen["B"].path_geometry)
print(f"Luftlinie {de(luftlinie)} m | größter Abstand zwischen den Routen A und B: {de(abstand)} m")
im_wasserschutz = layer["wasserschutz"].union_all().contains(Point(SOURCES[1]))
print("SOURCES[1] =", SOURCES[1], "liegt im Wasserschutzgebiet:", "ja" if im_wasserschutz else "nein")

# Gegenprobe: jede Route auf beiden Rastern bewerten (die Diagonale ist die Zahl von PathFinder).
# route_metrics erwartet das Suchfenster als Raster; hier ist das Fenster das ganze Raster.
kreuz = {}
for rv in raster:
    bewertung = route_metrics(routen, raster[rv], AUFLOESUNG)   # Route -> Länge und Kosten auf diesem Raster
    kreuz[f"auf Raster {rv} [EUR]"] = [round(bewertung[k]["construction_cost"]) for k in routen]
print(pd.DataFrame(kreuz, index=[f"Route {k}" for k in routen]).to_string())

fig, ax = plt.subplots(figsize=(8, 6))
bild = zeige(ax, raster["A"], "Zwei Routen, gleiche Punkte (Hintergrund: Raster A)", alpha=0.5)
farbleiste(fig, bild, ax)
for v, farbe in (("A", "tab:blue"), ("B", "tab:green")):
    p = routen[v]
    x, y = p.path_geometry.xy
    ax.plot(x, y, color=farbe, lw=2.2,
            label=f"Variante {v}: {de(p.total_length)} m, {de(p.total_cost / 1000)} kEUR")
ax.plot(*SOURCE, "o", ms=9, mfc="white", mec="black", label="SOURCE")
ax.plot(*TARGET, "s", ms=9, mfc="white", mec="black", label="TARGET")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=2, fontsize=8)   # unter der Karte, verdeckt keine Route
plt.show()

### Was die Tabellen zeigen

- Route B ist deutlich länger als Route A, hat aber fast keine Meter im Wald (Spalte "davon Wald"). Route A läuft ein Stück über den bewaldeten Hang und ist kürzer.
  B nimmt den Umweg über den Talboden, weil Wald in Variante B viel teurer ist.
- Die Kosten in der ersten Tabelle stammen aus zwei verschiedenen Kostentabellen. Dass Route B dort billiger aussieht, sagt nichts über die Routen selbst.
- Die zweite Tabelle vergleicht fair: Auf Raster A ist Route A billiger als Route B, auf Raster B ist Route B billiger als Route A.
  Jede Route ist in ihrer eigenen Kostenwelt die beste.
- Die Annahmen in der Kostentabelle entscheiden also mit über die Route, nicht nur die Geländedaten: Eine Kostentabelle ist eine planerische Entscheidung, und wir sollten sie offenlegen.

### Kurzform: alles in einem `PathFinder`-Aufruf

Wer das Raster nicht speichern will, gibt `PathFinder` die Kostentabelle und die Layer direkt mit:
`cost_assumptions` für den Basis-Layer und `datasets_to_modify` für die modifizierenden Layer (dieselben Angaben wie bei `modify_raster_from_dataset`).
Der Puffer `search_space_buffer_m` ist der Abstand um die Luftlinie, in dem gesucht wird. Die Diagonale unseres 5-km-Fensters ist 5 km × √2, also etwa 7,07 km;
mit 7100 m ist das Suchfenster daher das ganze Fenster, die Ausgabe unten zeigt die Rastergröße.
Wir nehmen die Dateien aus dem Cache, also ohne Netz, und vergleichen mit Variante A.

In [ ]:
layer_a = [{"input_data": layer_datei(name), "cost_assumptions": wert, "multiply": mult}
           for name, (wert, mult) in SCHUTZ_A.items() if len(layer[name]) > 0]
layer_a += [{"input_data": strassen, "cost_assumptions": 0.8, "multiply": True, "geometry_buffer_m": 10},
            {"input_data": sperrflaeche, "cost_assumptions": VERBOTEN}]

kurz = PathFinder(dataset_source=layer_datei("alkis"),           # Basis-Layer aus dem Cache
                  source_coords=SOURCE, target_coords=TARGET,
                  cost_assumptions={"unterart_name": KOSTEN_A},
                  datasets_to_modify=layer_a,
                  resolution_in_m=AUFLOESUNG,                    # wie oben
                  on_thin_features="ignore",                     # wie oben: keine lange Warnung
                  search_space_buffer_m=7100,                    # etwas mehr als die Diagonale des Fensters
                  neighborhood_str="r2")
pfad_kurz = kurz.find_route()
zeilen_kurz, spalten_kurz = kurz.geo_rasterizer.raster.shape
print(f"Raster der Kurzform: {zeilen_kurz} × {spalten_kurz} Zellen")
print(f"Kurzform: {de(pfad_kurz.total_length)} m, {de(pfad_kurz.total_cost)} EUR")
print(f"Variante A oben: {de(routen['A'].total_length)} m, {de(routen['A'].total_cost)} EUR")

### Übung Teil 3

Die nächsten Abschnitte brauchen `RASTER_A`, `RASTER_B`, `SOURCE` und `TARGET` unverändert. Arbeitet in der Übung deshalb mit neuen Namen und neuen Dateien.

1. **Preise ändern:** Kopiert die Tabelle (`eigene_kosten = dict(KOSTEN_A)`) und setzt alle Wald-Arten (`"Laubholz"`, `"Nadelholz"`, `"Laub- und Nadelholz"`, `"Gehölz"`)
   auf 500 statt 365. Baut mit `baue_raster(eigene_kosten, SCHUTZ_A, 0.8, "data/uebung_1.tiff")` ein neues Raster und rechnet die Route von `SOURCE` nach `TARGET`
   noch einmal (neuer `PathFinder` auf `data/uebung_1.tiff`). Wohin wandert die Route, und wie viele Meter laufen jetzt durch den Wald?
2. **Eigene Layer:** Baut ein Raster mit `basis_raster(KOSTEN_A)` und wendet `schutz_anwenden` und `eigene_layer_anwenden` an.
   Legt dann eine vorhandene Leerrohrtrasse darüber, zum Beispiel `LineString([(428500, 5357000), (429500, 5356300), (430250, 5355800)])`:
   ein `GeoDataFrame` mit `crs=CRS`, `geometry_buffer_m=4`, `cost_assumptions=20` und `ignore_value=None`, damit die Leerrohrtrasse auch gesperrte Zellen quert.
   Speichert unter einem neuen Namen (`save_raster`) und rechnet die Route neu: Wie stark zieht die Leerrohrtrasse die Route an? Probiert danach eine zweite Sperrfläche
   quer über Route A, zum Beispiel `box(429300, 5355500, 429700, 5355800)`.
3. **Schutzgebiete treffen:** Route A läuft ein Stück durch Schutzgebiet (siehe erste Tabelle). Nehmt `ziel_nw = SOURCES[1]`, einen Punkt im Wasserschutzgebiet im Nordwesten
   (die Ausgabe oben bestätigt, dass er dort liegt), und rechnet mit je einem neuen `PathFinder` auf `RASTER_A` und `RASTER_B` von `SOURCE` nach `ziel_nw`.
   Messt die Meter im Schutzgebiet wie in der Zelle oben (`intersection`): Wo laufen A und B jetzt?

---
# Teil 4: Dijkstra und Delta-Stepping

**Warum zwei Verfahren?** Die Suche ist der Rechenkern von PYORPS. Auf einem Raster mit Millionen Zellen entscheidet
das Verfahren darüber, wie lange wir warten, vor allem wenn viele Routen zu rechnen sind und mehrere Prozessorkerne zur
Verfügung stehen. Ihr solltet deshalb wissen, was hinter `algorithm="..."` steckt.

Beide Verfahren lösen dieselbe Aufgabe und liefern **dieselben exakten Kosten**: die billigste Route von einer Quelle
zu allen anderen Zellen. Sie unterscheiden sich darin, **in welcher Reihenfolge** sie die Zellen abarbeiten.

## Dijkstra: immer die nächste Zelle

Eine Warteschlange (*Heap*) hält Zellen mit ihrer vorläufigen Distanz von der Quelle. Dijkstra nimmt **immer die Zelle
mit der kleinsten Distanz** heraus. Diese Distanz ist dann endgültig. Danach prüft das Verfahren die Nachbarn: Ist die
Verbindung über die gerade fertige Zelle billiger, wird deren Distanz verbessert (*Relaxieren*).

- Jede Zelle wird **genau einmal** abgeschlossen, streng in der Reihenfolge der Distanz.
- Einfach und exakt, aber **sequenziell**: Welche Zelle als nächste kommt, hängt vom Ergebnis der vorigen ab.
- Gibt es nur ein Ziel, kann die Suche stoppen, sobald dieses Ziel abgeschlossen ist.

## Delta-Stepping: ganze Eimer auf einmal

Die Zellen werden nach Distanz in **Eimer** (*Buckets*) der Breite Δ (`delta`) sortiert: Eimer *k* enthält Zellen mit
vorläufiger Distanz in [*k*·Δ, (*k*+1)·Δ). Das Verfahren leert immer den **kleinsten nicht leeren Eimer** und bearbeitet
**alle Zellen darin gleichzeitig**, also parallel auf mehreren Prozessorkernen oder auf der Grafikkarte.

- *Kurze* Kanten (Gewicht ≤ Δ) können eine Zelle wieder in denselben Eimer legen und werden deshalb wiederholt
  relaxiert, bis der Eimer leer bleibt. *Lange* Kanten werden einmal nach dem Eimer relaxiert.
- Am Ende stehen dieselben exakten Distanzen wie bei Dijkstra, aber viele Zellen pro Schritt.
- **Δ ist ein Stellrad:** sehr kleines Δ verhält sich fast wie Dijkstra (wenig Parallelität), sehr großes Δ gibt viel
  Parallelität, aber mehr Wiederholungsarbeit. In PYORPS ist `delta=100` voreingestellt.

## Die Unterschiede in einer Tabelle

| | Dijkstra | Delta-Stepping |
|---|---|---|
| Reihenfolge | eine Zelle nach der anderen, nach Distanz | ein Eimer nach dem anderen, darin alle Zellen zugleich |
| Parallel | nein | ja (OpenMP auf dem Prozessor, CUDA auf der GPU) |
| Ergebnis | exakt | exakt, dieselben Kosten |
| Stellgröße | keine | Eimerbreite Δ, Threadzahl |
| Eigenart | einfach, keine Einstellung nötig, rechnet auch mit Steigungskosten | bearbeitet viele Zellen zugleich; wie viel das bringt, hängt von Raster, Zahl der Ziele und Zahl der Kerne ab |
| Grenze in PYORPS | – | **keine Steigungskosten** (Teil 5): mit Geländemodell fällt PYORPS auf Dijkstra zurück und warnt |

**Welches Verfahren schneller ist, sagen wir nicht voraus.** Es hängt vom Raster, von der Zahl der Ziele und von der
Zahl der Kerne ab. **Wir messen es unten selbst** und lesen die Antwort aus den Tabellen. Sicher ist nur eines: Mit
Geländemodell läuft PYORPS immer mit Dijkstra.

### Ein Spielraster zum Anschauen

Wir rechnen auf einem kleinen Raster (25 × 25 Zellen, eine gesperrte Wand, ein teurer Streifen) beide Reihenfolgen nach.
Die **Distanzen sind in beiden Fällen gleich**; die Bilder zeigen, *wann* eine Zelle abgeschlossen wird.
(Vereinfachung: Wir zeigen für Delta-Stepping den Eimer, in dem die Zelle am Ende endgültig ist.)

In [ ]:
import heapq

import matplotlib.pyplot as plt
import numpy as np

# Kleines Spielraster: 1 = normal, 6 = teurer Streifen, unendlich = gesperrt
n = 25
kosten = np.ones((n, n))
kosten[8:17, 6] = np.inf            # eine gesperrte Wand
kosten[5:20, 12:15] = 6.0           # ein teurer Streifen
quelle = (12, 2)

schritte = [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]


def dijkstra(kosten, quelle):
    # Gibt Distanzfeld und Abschluss-Reihenfolge zurück
    dist = np.full(kosten.shape, np.inf)
    rang = np.full(kosten.shape, -1)
    dist[quelle] = 0.0
    heap = [(0.0, quelle)]
    zaehler = 0
    while heap:
        d, (r, c) = heapq.heappop(heap)
        if d > dist[r, c] or rang[r, c] >= 0:
            continue
        rang[r, c] = zaehler          # Zelle ist jetzt endgültig
        zaehler += 1
        for dr, dc in schritte:
            rr, cc = r + dr, c + dc
            if 0 <= rr < kosten.shape[0] and 0 <= cc < kosten.shape[1] and np.isfinite(kosten[rr, cc]):
                laenge = np.hypot(dr, dc)
                neu = d + laenge * 0.5 * (kosten[r, c] + kosten[rr, cc])
                if neu < dist[rr, cc]:
                    dist[rr, cc] = neu
                    heapq.heappush(heap, (neu, (rr, cc)))
    return dist, rang


dist, rang = dijkstra(kosten, quelle)
delta = 10.0
eimer = np.where(np.isfinite(dist), np.floor(dist / delta), np.nan)

erreichbar = int(np.isfinite(dist).sum())
n_eimer = int(np.nanmax(eimer)) + 1
print(f"Erreichbare Zellen: {erreichbar}")
print(f"Dijkstra:       {erreichbar} Einzelschritte (eine Zelle pro Schritt)")
print(f"Delta-Stepping: mindestens {n_eimer} Runden (Eimer) bei Δ = {delta:g}; im größten Eimer "
      f"{int(np.bincount(eimer[np.isfinite(eimer)].astype(int)).max())} Zellen zugleich "
      "(mit den Wiederholungen innerhalb eines Eimers sind es etwas mehr Runden)")

fig, achsen = plt.subplots(1, 3, figsize=(12, 3.8))
bild = np.where(np.isfinite(kosten), kosten, np.nan)
m0 = achsen[0].imshow(bild, cmap="YlOrBr", vmin=0, vmax=8)
achsen[0].set_title("Kosten je Zelle (weiß = gesperrt)")
plt.colorbar(m0, ax=achsen[0], fraction=0.046, label="Kosten")
r1 = np.where(rang >= 0, rang, np.nan)
m1 = achsen[1].imshow(r1, cmap="viridis")
achsen[1].set_title("Dijkstra: Nummer des Schritts")
plt.colorbar(m1, ax=achsen[1], fraction=0.046, label="Schritt")
m2 = achsen[2].imshow(eimer, cmap="viridis")
achsen[2].set_title(f"Delta-Stepping: Eimer (Δ = {delta:g})")
cb2 = plt.colorbar(m2, ax=achsen[2], fraction=0.046, label="Eimer")
from matplotlib.ticker import MaxNLocator
cb2.locator = MaxNLocator(integer=True)   # Eimer sind ganze Zahlen
cb2.update_ticks()
for a in achsen:
    a.plot(*quelle[::-1], "r*", ms=12)
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()
plt.show()

Links: das Spielraster mit der Quelle (roter Stern). In der Mitte erscheinen **viele einzelne Schritte** von innen
nach außen: Dijkstra geht Zelle für Zelle. Rechts sind es **wenige Eimer**: Alle Zellen mit ähnlicher Distanz haben
dieselbe Farbe und lassen sich gleichzeitig bearbeiten. Die Wand und der teure Streifen verformen beide Bilder gleich,
denn die Distanzen sind dieselben.

Jetzt rechnen wir mit PYORPS auf dem echten Kostenraster.

---
## Die Läufe mit PYORPS

Jetzt rechnen wir mit PYORPS auf dem Kostenraster `RASTER_A` aus Teil 3, **immer mit beiden Algorithmen**, und vergleichen
Kosten, Route und Rechenzeit. Alle Tabellen entstehen beim Ausführen der Zellen. Die Sekunden hängen von eurem Rechner ab:
von der Zahl der Kerne, davon, was nebenher läuft, und von der Rastergröße. Zählen sollen **eure** Zahlen auf **eurem** Laptop.

**Zeit:** Dieser Teil rechnet je nach Rechner einige Minuten. Die Zellen für Δ und Threads (Abschnitt „“) könnt ihr überspringen;
die Zusammenfassung am Ende läuft auch ohne sie.

## Welche Routing-Arten gibt es?

PYORPS wählt die Art automatisch aus der Zahl der Quellen und Ziele. Entscheidend für die Rechenzeit ist, **wie viele Suchen** dahinterstehen.
Eine Suche liefert die billigsten Kosten von **einer** Quelle zu allen Zellen. Das heißt *Single-Source Shortest Path* (SSSP).

| Art | Eingabe | Ergebnis | Was PYORPS rechnet |
|---|---|---|---|
| 1→1 | eine Quelle, ein Ziel | `Path` | **eine** Suche von der Quelle; sie hört auf, sobald das Ziel feststeht |
| 1→N | eine Quelle, N Ziele | `PathCollection` mit N Routen | **eine** Suche von der Quelle (SSSP); alle N Ziele werden aus ihr abgelesen |
| M→1 | M Quellen, ein Ziel | M Routen | **M** Suchen, eine pro Quelle |
| M→N | M Quellen, N Ziele | M·N Routen, Quelle für Quelle | **M** Suchen, eine pro Quelle; jede liest N Ziele ab |
| paarweise | `pairwise=True`, gleich viele Quellen und Ziele | N Routen (Quelle *i* zu Ziel *i*) | eine Suche **pro Paar** (Dijkstra fasst Paare zusammen, die einen Punkt teilen) |

**„“ heißt hier:** mehrere Quellen, aber *getrennte* Suchen. `find_route` startet nie eine einzige Suche, die von allen Quellen
zugleich losläuft. Die Mehr-Quellen-Rechnung bauen wir weiter unten selbst: für jede Zelle die Kosten zur **billigsten** Quelle.

Beide Algorithmen sind exakt. Die Kosten sollten also gleich sein. Das prüfen wir in jeder Zelle und **drucken das Ergebnis der Prüfung** (`gleiche Kosten: ja` oder `nein`).
Nur wenn es dort `nein` heißt, erklärt die Zelle, woran das liegen kann.

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from matplotlib.colors import ListedColormap
from rasterio.transform import rowcol

from pyorps import PathFinder, full_window_buffer_m

# Raster und Punkte aus Teil 3 (falls Teil 3 nicht ausgeführt wurde)
if not all(n in globals() for n in ("RASTER_A", "RASTER_B", "SOURCE", "TARGET", "SOURCES", "TARGETS")):
    RASTER_A = "data/cost_raster_a.tiff"
    RASTER_B = "data/cost_raster_b.tiff"
    SOURCE = (427558, 5357403)
    TARGET = (430288, 5355723)
    SOURCES = [SOURCE, (428238, 5358313), (428663, 5356873)]
    TARGETS = [TARGET, (431613, 5355083), (429853, 5355148), (431273, 5356048)]
VERBOTEN = 65535                      # Zellwert für gesperrte Zellen


def de(x, nd=0):
    # Zahl im deutschen Format für Sätze: 3.206 m, 3,5 s
    return f"{x:,.{nd}f}".replace(",", "§").replace(".", ",").replace("§", ".")


def dz(x, nd=0):
    # Zahl für Tabellen: Dezimalkomma, keine Tausenderpunkte
    return f"{x:.{nd}f}".replace(".", ",")


def de_e(x):
    # Kleine Zahlen in Schreibweise 1,6e-15 (deutsches Komma)
    return f"{x:.1e}".replace(".", ",")


# Threads für Delta-Stepping: höchstens drei Viertel der Kerne, damit der Rechner nebenbei benutzbar bleibt
KERNE = os.cpu_count() or 2
THREADS = max(1, int(KERNE * 0.75))
OMP_GRENZE = os.environ.get("OMP_NUM_THREADS", "")
if OMP_GRENZE.isdigit():              # eine vorgegebene Obergrenze gilt
    THREADS = min(THREADS, int(OMP_GRENZE))

TOLERANZ = 1e-6                       # erlaubter relativer Kostenunterschied bei Routen (Kosten aus den Zellen der Route)
TOLERANZ_FELD = 1e-5                  # bei Kostenfeldern größer, siehe unten
ALGOS = ("dijkstra", "delta-stepping")
ZEITEN, CPU, ERGEBNISSE = {}, {}, {}  # Sekunden, CPU-Zeit / Uhrzeit und Ergebnisse, gefüllt von den Zellen unten

ERKLAERUNG = ("  Erklärung: Dijkstra ist die Referenz. Delta-Stepping mit mehreren Threads kann in seltenen Fällen eine etwas teurere Route\n"
              "  liefern (ein Kommentar im Quellcode von PYORPS nennt diese Möglichkeit für den Kern von 1→1 bei zwei und mehr Threads). Das ist hier kein Fehler des Notebooks.\n"
              "  Rechnet mit `num_threads=1` nach, dann gleichen sich die Kosten.")

# Das Kostenraster einmal für die Bilder lesen
with rasterio.open(RASTER_A) as src:
    kosten_a = src.read(1)
    transform_a = src.transform
    EXTENT = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)


def neuer_finder(raster, nachbarschaft="r2"):
    # Das ganze Raster ist das Suchfenster (wie in Teil 3)
    return PathFinder(dataset_source=raster, source_coords=SOURCE, target_coords=TARGET,
                      search_space_buffer_m=full_window_buffer_m(raster), neighborhood_str=nachbarschaft)


FINDER = neuer_finder(RASTER_A)


def lauf(algo, source, target, finder=None, **kw):
    # Ein find_route-Aufruf; Quelle, Ziel und Einstellungen werden pro Aufruf übergeben
    finder = FINDER if finder is None else finder
    if algo == "delta-stepping":
        kw.setdefault("num_threads", THREADS)
    return finder.find_route(source=source, target=target, algorithm=algo, **kw)


def zeit(text, aufruf, key=None, wiederholungen=3):
    # Ein Aufwärm-Lauf, dann das Minimum aus drei Läufen. Der erste Aufruf kann langsamer sein: er lädt das Raster,
    # legt den Suchspeicher an und füllt die Caches des Prozessors (kompiliert wird nichts, die Kerne sind fertig übersetzt).
    t0 = time.perf_counter()
    aufruf()
    erst = time.perf_counter() - t0
    zeiten = []
    c0, w0 = time.process_time(), time.perf_counter()
    for _ in range(wiederholungen):
        t0 = time.perf_counter()
        ergebnis = aufruf()
        zeiten.append(time.perf_counter() - t0)
    cpu_faktor = (time.process_time() - c0) / (time.perf_counter() - w0)   # etwa 1 bei einem Thread, etwa 2 bei zwei
    sek = min(zeiten)
    print(f"{text:<46s} {de(sek, 3):>7s} s   (erster Aufruf {de(erst, 3)} s)")
    if key is not None:
        ZEITEN[key], CPU[key] = sek, cpu_faktor
    return sek, ergebnis


def als_liste(ergebnis):
    # Path (eine Route) oder PathCollection (mehrere Routen) als Liste von Path
    return [ergebnis] if hasattr(ergebnis, "total_cost") else list(ergebnis)


def pruefe_reihenfolge(routen, quellen, ziele, paarweise=False):
    # PYORPS liefert die Routen in Eingabereihenfolge, bei M→N Quelle für Quelle. Das prüfen wir an den Endpunkten.
    # (Diese Prüfung hängt nicht von Threads ab, deshalb darf sie hart sein.)
    erwartet = list(zip(quellen, ziele)) if paarweise else [(s, z) for s in quellen for z in ziele]
    assert len(routen) == len(erwartet), "falsche Zahl von Routen"
    for p, (s, z) in zip(routen, erwartet):
        assert np.allclose(p.source, s, atol=5) and np.allclose(p.target, z, atol=5), "Reihenfolge der Routen stimmt nicht"


def vergleiche(routen_d, routen_s):
    # Kosten beider Algorithmen: Arrays, größter relativer Unterschied, Zahl zellgleicher Routen
    kd = np.array([p.total_cost for p in routen_d])
    ks = np.array([p.total_cost for p in routen_s])
    rel = float(np.max(np.abs(kd - ks) / kd))
    gleich = sum(np.array_equal(a.path_indices, b.path_indices) for a, b in zip(routen_d, routen_s))
    return kd, ks, rel, gleich


def pruefe_kosten(name, kd, ks, tol=TOLERANZ):
    # Sichtbare Prüfung statt assert: Dijkstra (kd) ist die Referenz, Delta-Stepping (ks) wird verglichen
    kd, ks = np.asarray(kd, float), np.asarray(ks, float)
    diff = np.abs(kd - ks)
    rel = float(np.max(diff / kd))
    ok = rel < tol
    print(f"{name}: gleiche Kosten: {'ja' if ok else 'nein'}, größter Unterschied {de_e(float(diff.max()))} EUR "
          f"(relativ {de_e(rel)}, Toleranz {de_e(tol)})")
    if not ok:
        print(ERKLAERUNG)
    return ok


def hintergrund(ax):
    # Kostenraster in Grau, gesperrte Zellen rot
    bild = np.where(kosten_a == VERBOTEN, np.nan, kosten_a.astype(float))
    ax.imshow(bild, cmap="Greys", vmin=0, vmax=800, extent=EXTENT, alpha=0.55)
    gesperrt = np.ma.masked_where(kosten_a != VERBOTEN, kosten_a)
    ax.imshow(gesperrt, cmap=ListedColormap(["#b03030"]), extent=EXTENT, alpha=0.6)
    ax.set_xlabel("Rechtswert [m]", fontsize=9)
    ax.set_ylabel("Hochwert [m]", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.ticklabel_format(style="plain", useOffset=False)


print(f"Rechner: {KERNE} logische Kerne.")
print(f"Threads für Delta-Stepping: {THREADS} (drei Viertel von {KERNE} Kernen = {int(KERNE * 0.75)}; "
      f"Obergrenze durch OMP_NUM_THREADS: {OMP_GRENZE if OMP_GRENZE else 'nicht gesetzt'}).")
print(f"Raster: {kosten_a.shape[1]} × {kosten_a.shape[0]} Zellen, {de(100 * np.mean(kosten_a == VERBOTEN), 1)} % gesperrt.")

## 1→1: ein Paar, zwei Algorithmen

`PathFinder` hält das Raster und die Einstellungen. Quelle, Ziel und Algorithmus geben wir bei jedem `find_route`-Aufruf neu an
(`source=`, `target=`, `algorithm=`); das Raster wird dabei nicht neu geladen.
Eigene Einstellungen des Algorithmus (`delta`, `num_threads`) gehen ebenfalls als Schlüsselwort an `find_route`.
Einen Tippfehler wie `detla=3` zeigen wir unten: PYORPS bricht mit einer Fehlermeldung ab. Sie ist englisch; „“ heißt „“.

Die Hilfsfunktion `zeit` misst jeden Aufruf nach einem Aufwärm-Lauf dreimal und nimmt das Minimum.
Der erste Aufruf kann langsamer sein, weil er das Raster lädt, Speicher für die Suche anlegt und die Caches des Prozessors füllt (kompiliert wird nichts).
Die Angabe „“ zeigt, wie groß dieser Unterschied bei euch ist.

In [ ]:
zeilen, routen = [], {}
for algo in ALGOS:
    sek, p = zeit(f"1→1 {algo}", lambda a=algo: lauf(a, SOURCE, TARGET), key=("1→1", algo))
    routen[algo] = p
    zeilen.append((algo, dz(p.total_cost, 2), dz(p.total_length, 1), dz(sek, 3)))
ERGEBNISSE["1→1"] = {a: [routen[a]] for a in ALGOS}

print()
print(pd.DataFrame(zeilen, columns=["Algorithmus", "Kosten [EUR]", "Länge [m]", "Sekunden"]).to_string(index=False))
print()

d, s = routen["dijkstra"], routen["delta-stepping"]
pruefe_kosten("1→1", [d.total_cost], [s.total_cost])

gleich = np.array_equal(d.path_indices, s.path_indices)
abstand = d.path_geometry.hausdorff_distance(s.path_geometry)
print(f"Dieselben Zellen: {'ja' if gleich else 'nein'}; größter Abstand der beiden Linien: {de(abstand, 1)} m")

# Ein Tippfehler im Namen einer Einstellung: PYORPS meldet ihn (die Meldung ist englisch)
print("\nTippfehler-Test mit `detla=3`. Die englische Meldung nennt den falschen Namen und schlägt den richtigen vor:")
try:
    FINDER.find_route(source=SOURCE, target=TARGET, algorithm="delta-stepping", detla=3)
    print("  keine Fehlermeldung (in dieser PYORPS-Version wird der Name nicht geprüft)")
except TypeError as fehler:
    print(f"  {type(fehler).__name__}: {fehler}")

**Wie lesen wir das?** Die Kosten einer Route berechnet `find_route` am Ende aus den Zellen der Route, bei beiden Algorithmen gleich.
Deshalb stimmen sie so genau überein. Steht oben bei „“ ein *nein*, dann haben die Algorithmen **zwei gleich teure Routen** gefunden.
Das kommt auf einem Raster mit großen gleichförmigen Flächen (Wald, Acker) oft vor: Dort kosten viele Folgen von Schritten genau gleich viel.
Welche von zwei gleich billigen Routen zuerst gefunden wird, hängt von der Reihenfolge ab, in der die Nachbarzellen abgearbeitet werden.
Eine Route ist also nicht eindeutig, **ihre Kosten schon**.

Steht oben bei „“ ein *nein*, dann ist Dijkstra die Referenz (siehe die Erklärung, die die Zelle dann druckt).

## 1→N: eine Quelle, vier Ziele

Hier lohnt sich die Idee der Single-Source-Suche: **Eine** Suche von `SOURCE` liest alle vier Ziele ab. Die Tabelle zeigt die Kosten pro Ziel
für beide Algorithmen. Das Bild zeigt die vier Routen auf dem Kostenraster (grau, gesperrte Zellen rot): Dijkstra durchgezogen,
Delta-Stepping gestrichelt darüber.

In [ ]:
sek = {}
pc = {}
for algo in ALGOS:
    sek[algo], pc[algo] = zeit(f"1→N {algo}", lambda a=algo: lauf(a, SOURCE, TARGETS), key=("1→N", algo))
    pruefe_reihenfolge(als_liste(pc[algo]), [SOURCE], TARGETS)
ERGEBNISSE["1→N"] = {a: list(pc[a]) for a in ALGOS}

kd, ks, rel, gleich = vergleiche(ERGEBNISSE["1→N"]["dijkstra"], ERGEBNISSE["1→N"]["delta-stepping"])
tabelle = pd.DataFrame({
    "Ziel": [f"Ziel {j + 1} (TARGETS[{j}])" for j in range(len(TARGETS))],
    "Länge [m]": [dz(p.total_length, 1) for p in ERGEBNISSE["1→N"]["dijkstra"]],
    "Kosten Dijkstra [EUR]": [dz(x, 2) for x in kd],
    "Kosten Delta-Stepping [EUR]": [dz(x, 2) for x in ks],
})
print()
print(tabelle.to_string(index=False))
print()
pruefe_kosten("1→N", kd, ks)
print(f"zellgleiche Routen: {gleich} von {len(kd)}")

# Verhältnis: eine Suche für vier Ziele gegenüber einer Suche für ein Ziel
for algo in ALGOS:
    print(f"{algo}: 1→N dauert das {de(ZEITEN[('1→N', algo)] / ZEITEN[('1→1', algo)], 1)}-fache von 1→1 (vier Ziele, eine Suche)")

fig, ax = plt.subplots(figsize=(8, 6))
hintergrund(ax)
for j, (pd_, ps_) in enumerate(zip(ERGEBNISSE["1→N"]["dijkstra"], ERGEBNISSE["1→N"]["delta-stepping"])):
    farbe = f"C{j}"
    ax.plot(*pd_.path_geometry.xy, color=farbe, lw=3, label=f"Ziel {j + 1}: {de(pd_.total_length)} m, {de(pd_.total_cost / 1000)} kEUR")
    ax.plot(*ps_.path_geometry.xy, color="black", lw=0.8, ls="--")
    ax.plot(*TARGETS[j], "s", ms=8, mfc="white", mec="black")
    ax.annotate(str(j + 1), TARGETS[j], xytext=(7, 7), textcoords="offset points", fontsize=10, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="black", lw=0.5))
ax.plot(*SOURCE, "o", ms=10, mfc="white", mec="black", label="SOURCE")
ax.set_title("1→N: vier Routen von SOURCE (durchgezogen: Dijkstra, gestrichelt: Delta-Stepping)", fontsize=9)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2, fontsize=8)   # unter der Karte, verdeckt keine Route
plt.tight_layout()
plt.show()

## M→N und paarweise: viele Quellen

Mit drei Quellen (`SOURCES`) und vier Zielen (`TARGETS`) entstehen **12 Routen**. PYORPS startet drei Suchen, eine pro Quelle, und liest jeweils vier Ziele ab.
Die Routen kommen Quelle für Quelle zurück; `pruefe_reihenfolge` bestätigt das an den Endpunkten der Routen. Die Matrix unten hat deshalb
eine Zeile pro Quelle und eine Spalte pro Ziel.

Danach folgen zwei kleinere Fälle: **M→1** (drei Quellen, ein Ziel) und **paarweise** (die drei Quellen mit den ersten drei Zielen, also nur drei Routen).
Bei „“ geben wir `num_threads` mit, PYORPS **ignoriert** es dort aber; die Zelle zeigt das am Ende mit der Spalte *CPU/Uhr*.

In [ ]:
pc = {}
for algo in ALGOS:
    _, pc[algo] = zeit(f"M→N {algo} (12 Routen)", lambda a=algo: lauf(a, SOURCES, TARGETS), key=("M→N", algo))
    pruefe_reihenfolge(list(pc[algo]), SOURCES, TARGETS)
ERGEBNISSE["M→N"] = {a: list(pc[a]) for a in ALGOS}

kd, ks, rel, gleich = vergleiche(ERGEBNISSE["M→N"]["dijkstra"], ERGEBNISSE["M→N"]["delta-stepping"])
form = (len(SOURCES), len(TARGETS))
zeilen_namen = [f"SOURCES[{i}]" for i in range(form[0])]
spalten_namen = [f"TARGETS[{j}]" for j in range(form[1])]
for name, werte in (("Dijkstra", kd), ("Delta-Stepping", ks)):
    print(f"\nKosten {name} [EUR]")
    print(pd.DataFrame(werte.reshape(form), index=zeilen_namen, columns=spalten_namen).to_string(float_format=lambda x: f"{x:.0f}"))

print()
pruefe_kosten("M→N (12 Routen)", kd, ks)
print(f"zellgleiche Routen: {gleich} von {len(kd)}")

In [ ]:
def beide(modus, text, quellen, ziele, paarweise=False):
    # Einen Fall mit beiden Algorithmen rechnen; Reihenfolge der Routen prüfen, Kosten sichtbar vergleichen
    ERGEBNISSE[modus] = {}
    ziel_liste = [ziele] if isinstance(ziele, tuple) else list(ziele)
    for algo in ALGOS:
        _, ergebnis = zeit(f"{modus} {algo} ({text})", lambda a=algo: lauf(a, quellen, ziele, pairwise=paarweise),
                           key=(modus, algo))
        ERGEBNISSE[modus][algo] = list(ergebnis)
        pruefe_reihenfolge(ERGEBNISSE[modus][algo], quellen, ziel_liste, paarweise=paarweise)
    kd, ks, rel, _ = vergleiche(ERGEBNISSE[modus]["dijkstra"], ERGEBNISSE[modus]["delta-stepping"])
    return kd, ks


kd1, ks1 = beide("M→1", "3 Routen", SOURCES, TARGET)                                   # drei Quellen, ein Ziel
kdp, ksp = beide("paarweise", "3 Routen", SOURCES, TARGETS[:3], paarweise=True)        # SOURCES[i] nach TARGETS[i]

print()
print(pd.DataFrame({
    "M→1: Quelle → TARGET": [f"SOURCES[{i}]" for i in range(len(SOURCES))],
    "Kosten Dijkstra [EUR]": [f"{x:.0f}" for x in kd1], "Kosten Delta-Stepping [EUR]": [f"{x:.0f}" for x in ks1],
}).to_string(index=False))
pruefe_kosten("M→1", kd1, ks1)
print()
print(pd.DataFrame({
    "paarweise: Paar": [f"SOURCES[{i}] → TARGETS[{i}]" for i in range(len(SOURCES))],
    "Kosten Dijkstra [EUR]": [f"{x:.0f}" for x in kdp], "Kosten Delta-Stepping [EUR]": [f"{x:.0f}" for x in ksp],
}).to_string(index=False))
pruefe_kosten("paarweise", kdp, ksp)

# Hat num_threads bei „“ eine Wirkung? CPU/Uhr zeigt, wie viele Threads wirklich arbeiten.
print()
zeit("paarweise delta-stepping, num_threads=1 verlangt",
     lambda: lauf("delta-stepping", SOURCES, TARGETS[:3], pairwise=True, num_threads=1), key=("paarweise", "1 Thread verlangt"))
print(f"CPU/Uhr bei verlangtem 1 Thread: {de(CPU[('paarweise', '1 Thread verlangt')], 1)}; "
      f"mit {THREADS} verlangten Threads: {de(CPU[('paarweise', 'delta-stepping')], 1)}. "
      f"Wäre `num_threads` wirksam, läge die erste Zahl nahe 1,0.")

## Das Kostenfeld: Kosten von der Quelle zu **jeder** Zelle

Bisher haben wir nur Routen gelesen. Die Suche selbst kennt aber mehr: den Preis jeder Zelle. Dieses *Kostenfeld* liefert `finder.cost_field(punkt, algorithm=...)`
(Kostenfelder gibt es in PYORPS seit Version 0.4, sie sind als *experimentell* markiert).
`field_array()` liefert das Feld als Array in der Form des Suchfensters, hier das ganze Raster (1000 × 1000 Zellen). Zellen, die nicht erreichbar sind, stehen auf `inf`.
Beide Algorithmen können das Feld rechnen, auch Dijkstra: Er läuft dafür bis zum Ende und hört nicht beim ersten Ziel auf.

Die **Mehr-Quellen-Rechnung** bauen wir selbst: ein Feld pro Quelle, dann in jeder Zelle das **Minimum**. Das ist genau das Ergebnis einer Suche,
die von allen Quellen zugleich startet. Die Isolinien verbinden Zellen mit gleichen Kosten. Darüber liegt die Route `SOURCE`→`TARGET` aus der ersten Rechnung.
Die Kosten sind in EUR wie bei den Routen.

**Warum ist die erlaubte Abweichung bei Feldern größer als bei Routen?** Die Kosten einer Route werden aus ihren Zellen neu berechnet. Ein Feld wird nicht neu berechnet,
es enthält die Werte der Suche selbst. Die PYORPS-Dokumentation zu Kostenfeldern nennt für Delta-Stepping 32-Bit-Zahlen als Grund für kleine Abweichungen gegenüber Dijkstra.
Wir erwarten bei Feldern deshalb einen kleinen Unterschied (relativ etwa 1e-6 bis 1e-5). Für Felder gilt die Toleranz 1e-5, für Routen 1e-6.
Die Zelle druckt zur Einordnung den Abstand zweier 32-Bit-Zahlen in dieser Größe. Er allein erklärt den größten Unterschied nicht ganz: Der Unterschied ist ein Vielfaches davon.
Die Reihenfolge der Summation entlang der Route kommt vermutlich dazu (nicht geprüft).

In [ ]:
def feld(punkt, algo):
    # Kostenfeld ab einem Punkt: (Zeilen, Spalten), in EUR, inf = nicht erreichbar
    kw = {"num_threads": THREADS} if algo == "delta-stepping" else {}
    with FINDER.cost_field(punkt, algorithm=algo, **kw) as cf:
        return cf.field_array(np.float64)


einzel, mehr = {}, {}
for algo in ALGOS:
    _, einzel[algo] = zeit(f"Kostenfeld ab SOURCE {algo}", lambda a=algo: feld(SOURCE, a), key=("Kostenfeld", algo))
    t0 = time.perf_counter()
    mehr[algo] = np.minimum.reduce([feld(q, algo) for q in SOURCES])
    print(f"{'Minimum über 3 Quellen ' + algo:<46s} {de(time.perf_counter() - t0, 3):>7s} s   (drei Felder, ein Lauf)")

# Beide Algorithmen: dasselbe Feld? (sichtbare Prüfung, kein assert)
print()
for name, fd, fs in (("ab SOURCE", einzel["dijkstra"], einzel["delta-stepping"]),
                     ("Minimum über SOURCES", mehr["dijkstra"], mehr["delta-stepping"])):
    beide_erreichbar = np.isfinite(fd) & np.isfinite(fs)
    verschieden = int((np.isfinite(fd) != np.isfinite(fs)).sum())
    max_abs = float(np.max(np.abs(fd[beide_erreichbar] - fs[beide_erreichbar])))
    kmax = float(fd[beide_erreichbar].max())
    rel = max_abs / kmax
    ok = verschieden == 0 and rel < TOLERANZ_FELD
    print(f"Feld {name}: {de(int(beide_erreichbar.sum()))} erreichbare Zellen, {verschieden} Zellen nur von einem Algorithmus erreicht; "
          f"gleiche Kosten: {'ja' if ok else 'nein'}, größter Unterschied {de(max_abs, 2)} EUR bei Kosten bis {de(kmax)} EUR "
          f"(relativ {de_e(rel)}, Toleranz {de_e(TOLERANZ_FELD)})")
    if not ok:
        print(ERKLAERUNG)
vmax = float(np.max(einzel["dijkstra"][np.isfinite(einzel["dijkstra"])]))
print(f"Zur Einordnung: Zwei benachbarte 32-Bit-Zahlen in der Größe {de(vmax)} liegen etwa {de(float(np.spacing(np.float32(vmax))), 3)} auseinander.")

# Das Feld am Ziel ist der Preis der Route
zeile, spalte = rowcol(transform_a, *TARGET)
print(f"Feld am Ziel: {de(einzel['dijkstra'][zeile, spalte], 2)} EUR; Routenkosten (1→1): {de(ERGEBNISSE['1→1']['dijkstra'][0].total_cost, 2)} EUR")

route = ERGEBNISSE["1→1"]["dijkstra"][0]
stufen = np.arange(200_000, vmax, 200_000)
fig, achsen = plt.subplots(1, 2, figsize=(9, 4.8), sharey=True)
for ax, f, titel in ((achsen[0], einzel["dijkstra"], "ab SOURCE"), (achsen[1], mehr["dijkstra"], "ab der billigsten der 3 Quellen")):
    bild = ax.imshow(np.where(np.isfinite(f), f, np.nan) / 1000, cmap="YlOrRd", vmin=0, vmax=vmax / 1000, extent=EXTENT)
    linien = ax.contour(np.where(np.isfinite(f), f, np.nan) / 1000, levels=stufen / 1000, colors="black", linewidths=0.6, extent=EXTENT, origin="upper")
    ax.clabel(linien, fmt="%d", fontsize=8)
    ax.plot(*route.path_geometry.xy, color="tab:blue", lw=2, label="Route SOURCE→TARGET")
    ax.plot(*zip(*SOURCES), "o", ms=7, mfc="white", mec="black", label="Quellen")
    ax.plot(*TARGET, "s", ms=7, mfc="white", mec="black", label="TARGET")
    ax.set_title(f"Kosten {titel} (Isolinien alle 200 kEUR)", fontsize=9)
    ax.set_xlabel("Rechtswert [m]", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.ticklabel_format(style="plain", useOffset=False)
    balken = plt.colorbar(bild, ax=ax, fraction=0.046, pad=0.02)
    balken.set_label("Kosten [kEUR]", fontsize=9)
    balken.ax.tick_params(labelsize=8)
achsen[0].set_ylabel("Hochwert [m]", fontsize=9)
achsen[0].legend(loc="lower left", fontsize=8, framealpha=0.9)
plt.tight_layout()
plt.show()

## Vertiefung (optional): die Stellschrauben `delta` und `num_threads`

Dieser Abschnitt ist freiwillig. Die beiden Zellen brauchen am meisten Zeit; wer sie überspringt, kann bei der Zusammenfassung direkt weitermachen.

Beide Einstellungen gehen als Schlüsselwort an `find_route`. Wir nehmen den M→N-Fall (12 Routen) und ändern jeweils **eine** Größe.

**Was ist Δ (`delta`) genau?** Die Einheit sind die **Zellwerte des Rasters** (hier EUR pro Meter), nicht Meter und nicht EUR.
Das Kantengewicht eines Schritts ist der **Mittelwert der Zellwerte entlang des Schritts mal die Länge des Schritts in Zellen**:
ein waagerechter Schritt zwischen zwei Zellen mit dem Wert 365 wiegt 365, ein diagonaler Schritt 365·√2.
Die gemeldeten Kosten in EUR sind dieses Gewicht mal die Zellgröße (5 m). **Δ = 100 sind also 500 EUR Routenkosten**, und das ist die Voreinstellung.
Eine Kante gilt als *kurz* (wird im Eimer wiederholt relaxiert), wenn ihr Gewicht höchstens Δ beträgt, sonst als *lang*.
Die Zelle druckt zuerst Median und Mittelwert der freien Zellen von `RASTER_A`. Die Tabelle zeigt in der zweiten Spalte, welcher Anteil der freien Zellen überhaupt höchstens Δ kostet
(eine Faustregel, denn ein Schritt über mehrere Zellen wiegt mehr).

`num_threads` ist die Zahl der Threads (Rechenstränge) auf dem Prozessor. Die Spalte *CPU/Uhr* ist die verbrauchte Rechenzeit geteilt durch die Uhrzeit:
etwa 1 heißt, es arbeitet ein Kern; etwa 2 heißt, es arbeiten zwei. So sehen wir **unabhängig vom Zeitrauschen**, ob die Threads wirklich laufen.

In [ ]:
frei = kosten_a[kosten_a != VERBOTEN]
print(f"Freie Zellen von RASTER_A: Median {de(np.median(frei))}, Mittelwert {de(frei.mean(), 1)}.")
referenz = np.array([p.total_cost for p in ERGEBNISSE["M→N"]["dijkstra"]])


def kosten_von(pc):
    return np.array([p.total_cost for p in pc])


# 1) Eimerbreite Δ, feste Threadzahl
zeilen = []
for delta in (10, 100, 1000, 10000):
    key = ("delta", delta)
    sek, pc = zeit(f"Δ = {delta:>5}, {THREADS} Threads", lambda d=delta: lauf("delta-stepping", SOURCES, TARGETS, delta=d), key=key)
    rel = float(np.max(np.abs(kosten_von(pc) - referenz) / referenz))
    zeilen.append((delta, f"{dz(100 * np.mean(frei <= delta), 1)} %", dz(sek, 3), dz(CPU[key], 2), de_e(rel), "ja" if rel < TOLERANZ else "nein"))
print()
print(pd.DataFrame(zeilen, columns=["Δ", "Zellen mit Wert ≤ Δ", "Sekunden", "CPU/Uhr", "max. rel. Kostenabweichung", "gleiche Kosten"]).to_string(index=False))
if any(z[-1] == "nein" for z in zeilen):
    print(ERKLAERUNG)

# 2) Threadzahl, Δ = 100
print()
zeilen = [("Dijkstra (ein Thread)", dz(ZEITEN[("M→N", "dijkstra")], 3), dz(CPU[("M→N", "dijkstra")], 2), "-", "Referenz")]
for n in sorted({t for t in (1, 2, THREADS) if t <= KERNE}):
    key = ("threads", n)
    sek, pc = zeit(f"Delta-Stepping, Δ = 100, {n} Thread(s)", lambda n=n: lauf("delta-stepping", SOURCES, TARGETS, num_threads=n), key=key)
    rel = float(np.max(np.abs(kosten_von(pc) - referenz) / referenz))
    zeilen.append((f"Delta-Stepping, {n} Thread(s)", dz(sek, 3), dz(CPU[key], 2), de_e(rel), "ja" if rel < TOLERANZ else "nein"))
ZEITEN[("M→N", "delta-1thread")] = ZEITEN[("threads", 1)]
print()
print(pd.DataFrame(zeilen, columns=["Variante", "Sekunden", "CPU/Uhr", "max. rel. Kostenabweichung", "gleiche Kosten"]).to_string(index=False))
if any(z[-1] == "nein" for z in zeilen):
    print(ERKLAERUNG)

**Wie lesen wir diese Tabellen?**

- **Die Ergebnisse sollten gleich sein.** In der Spalte „“ sollte bei jeder Einstellung *ja* stehen. `delta` und `num_threads` ändern **nur die Zeit**, nicht die Kosten.
  Steht dort *nein*, gilt Dijkstra als Referenz (siehe die Erklärung, die die Zelle dann druckt).
- **Wie viel sich die Zeit ändert, hängt von der Maschine ab:** von der Zahl der Kerne, davon, ob andere Programme Kerne belegen, und von der Größe des Rasters.
- **Allgemeine Erfahrung, hier nicht gemessen:** Auf einem Raster mit vielen Millionen Zellen lohnt sich der Aufwand für Eimer und Threads eher als auf diesem mit 1 Million Zellen.
- **Die Δ-Tabelle zeigt den Einfluss von Δ auf eurem Rechner:** Vergleicht die Sekunden bei Δ = 10 und Δ = 10000 mit denen bei Δ = 100 und Δ = 1000. Gemessen ist nur das.
  **Allgemeine Erfahrung, hier nicht gemessen** ist die Begründung: Bei sehr kleinem Δ gibt es viele Eimer mit wenigen Zellen (kaum Parallelität), bei sehr großem Δ wird viel doppelt relaxiert.
  Die Voreinstellung 100 ist ein Anfang, kein Optimum für jedes Raster.
- **Allgemeine Erfahrung, hier nicht gemessen:** Mehr Threads sind nicht immer schneller, weil sie in jedem Eimer aufeinander warten. Wir nehmen deshalb höchstens drei Viertel der Kerne (so macht es PYORPS bei Kostenfeldern selbst).
  Die Thread-Tabelle zeigt nur die Zahlen 1, 2 und `THREADS`.
- **Messt selbst:** Eure Zahlen auf eurem Laptop sind die, die zählen. Kleine Unterschiede können Rauschen sein; wiederholt die Messung, bevor ihr etwas daraus schließt.

## Zusammenfassung und Grenzen

Die Tabelle unten wird aus den Zeiten der Zellen oben gebaut. Die letzte Spalte ist **gerechnet**, nicht behauptet: Sie sagt, welcher Algorithmus auf **diesem** Rechner schneller war.
Die Zeile „“ gibt es nur, wenn ihr die Vertiefung ausgeführt habt; sie zeigt, wie viel vom Vorsprung nur von den Threads kommt und wie viel vom Verfahren selbst. Liegt der Faktor in dieser Zeile deutlich über 1, kommt der Vorsprung nicht nur von den Threads.
Die Zeile „“ ist mit einem Stern markiert: Dort ignoriert PYORPS `num_threads`, die Zeile ist darum **nicht vergleichbar** und zählt in der Auswertung nicht mit.

**Grenzen, die ihr kennen solltet:**

- **Keine Steigungskosten.** Delta-Stepping kennt nur Kosten aus dem Raster. Mit Geländemodell (Teil 5) läuft PYORPS stattdessen mit Dijkstra und warnt (`AlgorithmFallbackWarning`).
- **`pairwise=True` ignoriert `num_threads`.** Die Zelle zu „“ zeigt das mit der Spalte *CPU/Uhr*. Delta-Stepping läuft dort mit der Voreinstellung (alle Kerne, höchstens 12, oder die Obergrenze `OMP_NUM_THREADS`).
- **Threads und Betriebssystem.** Delta-Stepping nutzt mehrere Threads über OpenMP, auch unter Windows: Die Spalte *CPU/Uhr* oben zeigt, wie viele Kerne wirklich arbeiten.
  Unter macOS ist OpenMP laut `setup.py` nur dabei, wenn die Erweiterung mit Homebrew `libomp` gebaut wurde (von uns nicht getestet); wer einen Mac hat, prüft die Spalte *CPU/Uhr* selbst.
- **Mehr Quellen sind getrennte Suchen** (siehe Tabelle am Anfang), außer ihr baut das Minimum aus Kostenfeldern wie oben.

In [ ]:
modi = [("1→1", "1→1: SOURCE → TARGET"), ("1→N", "1→N: SOURCE → 4 Ziele"), ("M→1", "M→1: 3 Quellen → TARGET"),
        ("M→N", "M→N: 3 Quellen → 4 Ziele (12)"), ("paarweise", "paarweise: 3 Paare *"), ("Kostenfeld", "Kostenfeld ab SOURCE")]
zeilen, vergleichbar = [], []
for key, text in modi:
    d, s = ZEITEN[(key, "dijkstra")], ZEITEN[(key, "delta-stepping")]
    threads = f"Standard (CPU/Uhr {de(CPU[(key, 'delta-stepping')], 1)})" if key == "paarweise" else str(THREADS)
    zeilen.append((text, dz(d, 3), dz(s, 3), dz(d / s, 2), threads, "Delta-Stepping" if s < d else "Dijkstra"))
    if key != "paarweise":
        vergleichbar.append(s < d)
if ("M→N", "delta-1thread") in ZEITEN:        # nur, wenn die Vertiefung ausgeführt wurde
    d, s = ZEITEN[("M→N", "dijkstra")], ZEITEN[("M→N", "delta-1thread")]
    zeilen.append(("M→N mit einem Thread", dz(d, 3), dz(s, 3), dz(d / s, 2), "1", "Delta-Stepping" if s < d else "Dijkstra"))
    vergleichbar.append(s < d)

tabelle = pd.DataFrame(zeilen, columns=["Modus", "Dijkstra [s]", "Delta-Stepping [s]", "Faktor Dijkstra/Delta", "Threads Delta-Stepping", "schneller"])
print(tabelle.to_string(index=False))
print("* paarweise: PYORPS ignoriert dort num_threads und nimmt die Voreinstellung (CPU/Uhr zeigt die wirkliche Zahl). "
      "Die Zeile ist darum nicht mit den anderen vergleichbar und zählt unten nicht mit.")
print(f"\nRechner: {KERNE} logische Kerne, Delta-Stepping mit {THREADS} Threads (Obergrenze OMP_NUM_THREADS: {OMP_GRENZE if OMP_GRENZE else 'nicht gesetzt'}), "
      f"Δ = 100, Raster {de(kosten_a.size)} Zellen.")
print(f"Auf diesem Rechner war Delta-Stepping in {sum(vergleichbar)} von {len(vergleichbar)} vergleichbaren Zeilen schneller.")

### Übung Teil 4

1. **Nachbarschaft.** Baut mit `neuer_finder(RASTER_A, "r1")` und `neuer_finder(RASTER_A, "r3")` zwei neue `PathFinder` und rechnet `SOURCE` → `TARGET` mit beiden Algorithmen
   (`lauf(algo, SOURCE, TARGET, finder=...)`). Vergleicht Kosten, Länge und Zeit mit `"r2"`. Wie verändert die Nachbarschaft die Route, und wie die Zeit von Dijkstra im Vergleich zu Delta-Stepping?
2. **Zweites Raster.** Rechnet dasselbe Paar auf `RASTER_B` (`neuer_finder(RASTER_B)`) mit beiden Algorithmen. Stimmen die Kosten wieder überein? Vergleicht die Route mit Variante B aus Teil 3.
3. **Eigene Δ-Werte.** Rechnet den M→N-Fall mit `delta=30` und `delta=3000` und mit `num_threads=1` und `num_threads=2`. Bleiben die Kosten gleich? Wo liegt auf eurem Rechner das beste Δ?
   *Vorsicht:* Ein sehr großes Δ (zum Beispiel 100000) kann den Lauf deutlich verlangsamen. Probiert es mit einem einzelnen Aufruf, nicht mit `zeit`.

---
# Teil 5: Gelände berücksichtigen (DGM, 3D-Routing)

Bisher war die Welt flach: Ein Meter Strecke kostete überall gleich viel, egal wie steil es ist. Für Kabel und Leitungen stimmt das nicht.

**Was ist ein DGM?** Ein *digitales Geländemodell* (DGM) gibt für jede Rasterzelle die Höhe des nackten Erdbodens an. Ein *digitales Oberflächenmodell* (DOM, englisch DSM) enthält zusätzlich Bäume und Gebäude. Zum Verlegen eines Kabels brauchen wir den Boden, deshalb nehmen wir ein DGM. In pyorps heißt das Argument `dem=` (englisch *digital elevation model*); gemeint ist dasselbe.

**Warum ist die Neigung wichtig?**
- Am Hang ist der Graben schwerer zu bauen, und der Boden kann abrutschen oder erodieren.
- Steile Hänge sind für Baumaschinen schwer zugänglich.
- Eine Route, die über den Berg geht, ist länger als die Karte zeigt.

**Neigung in Prozent:** Hangneigung in % = Höhenunterschied / Strecke × 100. 20 % bedeutet 2 m Anstieg auf 10 m Strecke. In Formeln rechnen wir mit der Neigung als Verhältnis (0,2 = 20 %).

**2D- gegen 3D-Länge:** Die 2D-Länge misst nur im Grundriss. Die 3D-Länge misst am Boden entlang: Pro Schritt gilt `sqrt(Strecke² + Höhenunterschied²)`.

In diesem Teil lernen wir, wie `PathFinder` ein DGM nutzt, wie ihr das Gewicht der Neigung wählt, wie wir eine harte Grenze setzen und welche Fallen es gibt.

*Daten:* Die Datei `data/dem_clip.tif` (das DGM für unser Gebiet plus 300 m Rand, 5 m Zellen, EPSG:25832) liegt dem Repository bei. Nur wenn sie fehlt, lädt die Funktion `hole_dgm()` das DGM beim LGL herunter (etwa 235 MB in 16 Kacheln, das dauert länger als eine Minute); ohne Netz nimmt sie ein künstliches Gelände.

*Datenquelle:* Geländemodell: DGM1 Baden-Württemberg, © LGL, www.lgl-bw.de, Datenlizenz Deutschland – Namensnennung – Version 2.0 (dl-de/by-2-0, www.govdata.de/dl-de/by-2-0); Datensatz: https://opengeodata.lgl-bw.de (DGM1); von 1 m auf 5 m gemittelt.

In [ ]:
# Setup für Teil 5: Importe und Variablen aus Teil 3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from matplotlib.patches import Rectangle
from matplotlib.ticker import MaxNLocator
from scipy.ndimage import map_coordinates
from scipy.spatial import cKDTree

from pyorps import PathFinder, Objective, GradientOptions, AlgorithmFallbackWarning, NoPathFoundError, route_metrics

def de(x, nd=0):
    # Zahl im deutschen Format: Punkt als Tausendertrenner, Komma als Dezimalzeichen
    return f"{x:,.{nd}f}".replace(",", "§").replace(".", ",").replace("§", ".")

def tab_de(df, nd=1):
    # Tabelle mit Dezimalkomma und ohne Tausendertrenner ausgeben (nur Kommazahlen, ganze Zahlen bleiben)
    return df.apply(lambda spalte: spalte.map(lambda x: f"{x:.{nd}f}".replace(".", ",")) if spalte.dtype.kind == "f" else spalte)

# (falls Teil 3 nicht ausgeführt wurde): dieselben Werte wie dort
if "RASTER_A" not in globals():
    BBOX = (427000, 5354000, 432000, 5359000)   # (min Rechtswert, min Hochwert, max Rechtswert, max Hochwert)
    CRS = "EPSG:25832"
    AUFLOESUNG = 5                               # Meter pro Zelle
    RASTER_A = "data/cost_raster_a.tiff"
    RASTER_B = "data/cost_raster_b.tiff"
    SOURCE = (427558, 5357403)                   # Kinzigtal-Talboden
    TARGET = (430288, 5355723)                   # westliches Ende des Harmersbachtals
print("Kostenraster A:", Path(RASTER_A).resolve(), "| vorhanden:", Path(RASTER_A).exists())

## Das DGM beschaffen

Die nächste Zelle enthält die Funktion `hole_dgm()` und ein paar Hilfsfunktionen. Ihr müsst sie nur ausführen, nicht lesen.
`hole_dgm()` gibt den Pfad zu einer DGM-Datei zurück: Liegt `data/dem_clip.tif` schon da (so ist es im Repository), ist das sofort fertig. Nur wenn die Datei fehlt, lädt die Funktion das DGM1 beim LGL herunter (etwa 235 MB, 16 Kacheln, mehr als eine Minute) und mittelt es auf 5 m. Gelingt das nicht, nimmt sie ein künstliches Gelände und sagt es deutlich.

In [ ]:
# Hilfsfunktionen für das DGM (Erklärung in der Zelle davor)
import io
import os
import urllib.request
import warnings
import zipfile

import numpy as np

# Quellenvermerk (Pflicht nach dl-de/by-2-0)
DEM_QUELLE = (
    "Geländemodell: DGM1 Baden-Württemberg, © LGL, www.lgl-bw.de, "
    "Datenlizenz Deutschland – Namensnennung – Version 2.0 (dl-de/by-2-0, "
    "www.govdata.de/dl-de/by-2-0); Datensatz: https://opengeodata.lgl-bw.de (DGM1); "
    "von 1 m auf 5 m gemittelt"
)
DEM_QUELLE_SYNTHETISCH = "SYNTHETISCHES Gelände (numpy-Hügel), keine echten Höhen"

BBOX = (427000, 5354000, 432000, 5359000)  # EPSG:25832: xmin, ymin, xmax, ymax (Kinzigtal bei Biberach)
RAND_M = 300
AUFLOESUNG_M = 5
NODATA = -9999.0
LGL_URL = "https://opengeodata.lgl-bw.de/data/dgm/dgm1_32_{e}_{n}_2_bw.zip"
TIMEOUT_S = 30


def _ziel_gitter(bbox, rand_m, aufloesung_m):
    xmin, ymin, xmax, ymax = bbox
    xmin, ymin, xmax, ymax = xmin - rand_m, ymin - rand_m, xmax + rand_m, ymax + rand_m
    breite = int(round((xmax - xmin) / aufloesung_m))
    hoehe = int(round((ymax - ymin) / aufloesung_m))
    return xmin, ymax, breite, hoehe


def _schreibe_tif(pfad, z, xmin, ymax, aufloesung_m, quelle):
    import rasterio
    from rasterio.transform import from_origin

    os.makedirs(os.path.dirname(os.path.abspath(pfad)), exist_ok=True)
    profil = dict(
        driver="GTiff", height=z.shape[0], width=z.shape[1], count=1, dtype="float32",
        crs="EPSG:25832", transform=from_origin(xmin, ymax, aufloesung_m, aufloesung_m),
        nodata=NODATA, compress="deflate", predictor=3, tiled=True, blockxsize=256, blockysize=256,
    )
    tmp = pfad + ".part"
    with rasterio.open(tmp, "w", **profil) as dst:
        dst.write(z.astype("float32"), 1)
        dst.update_tags(QUELLE=quelle)
    os.replace(tmp, pfad)


def dgm_quelle(pfad):
    """Quellenvermerk einer mit hole_dgm() geschriebenen Datei (Tag QUELLE)."""
    import rasterio

    with rasterio.open(pfad) as src:
        return src.tags().get("QUELLE", "unbekannt")


def _lade_zip(url, versuche):
    for i in range(versuche):
        try:
            with urllib.request.urlopen(url, timeout=TIMEOUT_S) as r:
                return r.read()
        except Exception:  # Netzfehler: einmal erneut versuchen, dann aufgeben
            if i == versuche - 1:
                raise


def _dgm1_lgl(bbox, rand_m, aufloesung_m, url=LGL_URL):
    """DGM1-Kacheln laden und auf das Zielgitter mitteln (Blockmittel, aufloesung_m ganzzahlig)."""
    xmin, ymax, breite, hoehe = _ziel_gitter(bbox, rand_m, aufloesung_m)
    a = int(aufloesung_m)
    # 1-m-Mosaik des Zielfensters; NaN = keine Messung
    mosaik = np.full((hoehe * a, breite * a), np.nan, dtype="float32")
    x0, y1 = xmin, ymax
    x1, y0 = xmin + breite * a, ymax - hoehe * a
    # 1-km-Teilkacheln, die das Fenster berühren
    km_e = range(int(np.floor(x0 / 1000)), int(np.ceil(x1 / 1000)))
    km_n = range(int(np.floor(y0 / 1000)), int(np.ceil(y1 / 1000)))
    # 2-km-ZIPs: Ostwert ungerade, Nordwert gerade (so liefert das LGL-Portal die Kacheln)
    zips = sorted({(e if e % 2 else e - 1, n if n % 2 == 0 else n - 1) for e in km_e for n in km_n})
    for k, (e2, n2) in enumerate(zips):
        # erste Kachel ohne zweiten Versuch: ohne Netz scheitert die Zelle schnell
        z = zipfile.ZipFile(io.BytesIO(_lade_zip(url.format(e=e2, n=n2), 1 if k == 0 else 2)))
        for name in z.namelist():
            if not name.endswith(".xyz"):
                continue
            teile = os.path.basename(name).split("_")  # dgm1_32_<e>_<n>_1_bw_<jahr>.xyz
            e, n = int(teile[2]), int(teile[3])
            if e not in km_e or n not in km_n:
                continue
            xyz = np.loadtxt(io.BytesIO(z.read(name)), dtype="float64")
            spalte = np.floor(xyz[:, 0] - x0).astype(np.int64)
            zeile = np.floor(y1 - xyz[:, 1]).astype(np.int64)
            ok = (spalte >= 0) & (spalte < mosaik.shape[1]) & (zeile >= 0) & (zeile < mosaik.shape[0])
            mosaik[zeile[ok], spalte[ok]] = xyz[ok, 2]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)  # leere Blöcke -> NaN
        grob = np.nanmean(mosaik.reshape(hoehe, a, breite, a), axis=(1, 3))
    return np.where(np.isfinite(grob), grob, NODATA), xmin, ymax


def synthetisches_gelaende(bbox=BBOX, rand_m=RAND_M, aufloesung_m=AUFLOESUNG_M, seed=7):
    """Glattes Tal mit Hängen und Kuppen auf dem Zielgitter, Höhen ca. 180 bis 550 m.

    Ein ebener Talboden (ca. 600 m breit) läuft von Nord nach Süd durch die Mitte,
    die Hänge steigen nach beiden Seiten an, Gauß-Kuppen machen sie unregelmäßig.
    """
    xmin, ymax, breite, hoehe = _ziel_gitter(bbox, rand_m, aufloesung_m)
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:hoehe, 0:breite].astype("float64") * aufloesung_m  # Meter ab Nordwestecke
    achse = 0.5 * breite * aufloesung_m + 300.0 * np.sin(yy / 1500.0)      # geschwungene Talachse
    abstand = np.maximum(np.abs(xx - achse) - 300.0, 0.0)                 # 0 auf dem Talboden
    z = 180.0 + 5.0 * yy / 1000.0 + 280.0 * (1.0 - np.exp(-(abstand / 900.0) ** 2))
    for _ in range(15):
        cx = rng.uniform(0, breite * aufloesung_m)
        cy = rng.uniform(0, hoehe * aufloesung_m)
        r = rng.uniform(250.0, 700.0)
        kuppe = rng.uniform(20.0, 90.0) * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * r * r))
        z += kuppe * (1.0 - np.exp(-(abstand / 300.0) ** 2))  # Talboden bleibt eben
    return z, xmin, ymax


def hole_dgm(pfad="data/dem_clip.tif", bbox=BBOX, rand_m=RAND_M, aufloesung_m=AUFLOESUNG_M,
             download=True, synthetisch_pfad="data/dem_synthetisch.tif", url=LGL_URL):
    """Pfad zu einem DGM-GeoTIFF (EPSG:25832, float32, nodata=-9999) für bbox plus Rand."""
    if os.path.exists(pfad):
        return pfad
    if download:
        try:
            z, xmin, ymax = _dgm1_lgl(bbox, rand_m, aufloesung_m, url)
            _schreibe_tif(pfad, z, xmin, ymax, aufloesung_m, DEM_QUELLE)
            return pfad
        except Exception as fehler:
            warnings.warn(f"DGM-Download fehlgeschlagen ({fehler!r}); nehme synthetisches Gelände.")
    if not os.path.exists(synthetisch_pfad):
        z, xmin, ymax = synthetisches_gelaende(bbox, rand_m, aufloesung_m)
        _schreibe_tif(synthetisch_pfad, z, xmin, ymax, aufloesung_m, DEM_QUELLE_SYNTHETISCH)
    print("ACHTUNG: synthetisches Gelände, keine echten Höhen:", synthetisch_pfad)
    return synthetisch_pfad

# Datei holen: erst der Cache, sonst Download, sonst synthetisches Gelände
DEM = hole_dgm()
print("DGM-Datei:", Path(DEM).resolve())
print(DEM_QUELLE)

## Das Gelände ansehen

Wir lesen das DGM ein und rechnen die Neigung selbst aus (`numpy.gradient`, in Prozent). Das Schummerungsbild (*hillshade*) beleuchtet das Gelände schräg von Nordwesten. Es zeigt Hänge besser als Farben allein. Der Kreis ist `SOURCE`, das Quadrat `TARGET`, der cyanfarbene Rahmen das Untersuchungsgebiet.
Die Zahlen unten gelten für das Untersuchungsgebiet `BBOX`; das DGM reicht rundherum 300 m darüber hinaus.

Danach vergleichen wir die Neigung mit dem Kostenraster A: Wo es steil ist, liegt oft Wald.

In [ ]:
# DGM einlesen, Neigung in Prozent, Schummerung
with rasterio.open(DEM) as src:
    dem = src.read(1).astype("float64")
    T = src.transform                      # Transform des DGM (5 m, EPSG:25832)
    dem[dem == src.nodata] = np.nan
    quelle = src.tags().get("QUELLE", "unbekannt")
ZELLE = abs(T.a)
dy, dx = np.gradient(dem, ZELLE)
slope = np.hypot(dx, dy) * 100             # Neigung in Prozent
EXT = (T.c, T.c + T.a * dem.shape[1], T.f + T.e * dem.shape[0], T.f)   # links, rechts, unten, oben
hs = LightSource(azdeg=315, altdeg=45).hillshade(np.nan_to_num(dem, nan=np.nanmean(dem)),
                                                 vert_exag=1, dx=ZELLE, dy=ZELLE)

def zelle_von_xy(x, y):
    # Spalte und Zeile (ganzzahlig) im DGM-Raster
    return int((x - T.c) / T.a), int((y - T.f) / T.e)

c0, r1 = zelle_von_xy(BBOX[0], BBOX[1])
c1, r0 = zelle_von_xy(BBOX[2], BBOX[3])
im_gebiet = np.s_[r0:r1, c0:c1]            # Ausschnitt von BBOX im DGM

def achsen(ax):
    # Achsen in Metern, wenige Beschriftungen
    ax.xaxis.set_major_locator(MaxNLocator(4)); ax.yaxis.set_major_locator(MaxNLocator(4))
    ax.ticklabel_format(useOffset=False, style="plain")
    ax.tick_params(labelsize=8)
    ax.set_xlabel("Rechtswert [m]", fontsize=8); ax.set_ylabel("Hochwert [m]", fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].imshow(hs, cmap="gray", extent=EXT)
bild = axes[0].imshow(dem, cmap="terrain", alpha=0.45, extent=EXT)
fig.colorbar(bild, ax=axes[0], shrink=0.75, label="Höhe [m]")
axes[0].set_title("Gelände (Höhe und Schummerung)", fontsize=10)
bild = axes[1].imshow(slope, cmap="magma", vmin=0, vmax=60, extent=EXT)
fig.colorbar(bild, ax=axes[1], shrink=0.75, label="Hangneigung [%] (Skala bis 60)")
axes[1].set_title("Neigung", fontsize=10)
for ax in axes:
    ax.add_patch(Rectangle(BBOX[:2], BBOX[2] - BBOX[0], BBOX[3] - BBOX[1], fill=False, ec="cyan", lw=1))
    ax.plot(*SOURCE, "o", mfc="white", mec="black", ms=6, label="SOURCE")
    ax.plot(*TARGET, "s", mfc="white", mec="black", ms=6, label="TARGET")
    achsen(ax)
axes[0].legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

hoehe = dem[im_gebiet]; neigung = slope[im_gebiet]
print("Quelle der Höhen:", quelle)
print(f"Höhe im Untersuchungsgebiet: {de(np.nanmin(hoehe), 1)} bis {de(np.nanmax(hoehe), 1)} m")
print(f"Neigung in %: Median (50 %) = {de(np.nanpercentile(neigung, 50), 1)}, 90 %-Quantil = {de(np.nanpercentile(neigung, 90), 1)}, 99 %-Quantil = {de(np.nanpercentile(neigung, 99), 1)}")
print(f"Anteil der Zellen steiler als 15 %: {de(100 * np.mean(neigung > 15), 1)} %, flacher als 3 %: {de(100 * np.mean(neigung < 3), 1)} %")

In [ ]:
# Neigung gegen Kostenraster A: Anteil der Kostenklassen je Neigungsklasse
with rasterio.open(RASTER_A) as src:
    kosten = src.read(1)
    Tk = src.transform
# Kostenraster liegt im DGM-Raster (gleiche Zellgröße, gleiche Ausrichtung); wir schneiden aus der Neigung aus
ck, rk = zelle_von_xy(Tk.c, Tk.f)
assert abs(Tk.a) == ZELLE and ck >= 0 and rk >= 0
neigung_k = slope[rk:rk + kosten.shape[0], ck:ck + kosten.shape[1]]
assert neigung_k.shape == kosten.shape

neigungsklasse = pd.cut(neigung_k.ravel(), [0, 10, 20, 40, np.inf], right=False,
                        labels=["0-10 %", "10-20 %", "20-40 %", "über 40 %"])
kostenklasse = pd.cut(kosten.ravel().astype("float64"), [0, 150, 300, 65535, 70000], right=False,
                      labels=["günstig (< 150)", "mittel (150-299)", "teuer (300 und mehr, vor allem Wald)", "gesperrt (65535)"])
tab = (pd.crosstab(neigungsklasse, kostenklasse, normalize="index") * 100).rename_axis(index=None, columns=None)
tab.insert(0, "Zellen", pd.Series(neigungsklasse).value_counts().sort_index())
print("Anteil der Kostenklassen [%] je Neigungsklasse (Zeilen ergeben 100 %):")
print(tab_de(tab).to_string())

ax = tab.drop(columns="Zellen").plot.barh(stacked=True, figsize=(7, 2.8), colormap="viridis")
ax.set_xlabel("Anteil der Zellen [%]", fontsize=8); ax.set_ylabel("Hangneigung", fontsize=8); ax.tick_params(labelsize=8); ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

## Eine 3D-Route berechnen

Ein DGM wird mit `dem=` an `PathFinder` gegeben. Wie stark die Neigung zählt, legt ein `Objective` fest:

```python
Objective(weights={"cost": 1.0, "gradient": 5.0})
```

Die Suche minimiert dann `Kosten + Gewicht × Neigung`, gerechnet über jedes Stück der Route. Schon ohne Gewicht gilt nach dem pyorps-Quelltext: Mit DGM ist jedes Stück am Hang länger (3D-Streckung `sqrt(1 + Neigung²)`, Neigung als Verhältnis, 0,2 = 20 %; nicht einstellbar).

Wir testen auf einem **Höhenrücken-Paar**: Start auf dem Talboden der Kinzig, Ziel im Harmersbachtal, dazwischen ein bewaldeter Höhenrücken. Die Suche bekommt 1500 m Puffer um die Luftlinie.
Die Tabelle misst jede Route gleich: 2D-Länge, 3D-Länge, steilste Kante, höchster Punkt und Gesamtanstieg rechnen wir selbst aus den Höhen entlang der Route. `Kosten auf Raster A [EUR]` bewertet jede Route mit Raster A (Zellwert × 2D-Länge, `route_metrics` wie in Teil 3). Diese Spalte ist für 2D- und 3D-Routen dieselbe Messung. Das `total_cost` einer 3D-Route lassen wir weg: pyorps nennt es Einheiten der Zielfunktion, nicht EUR, und garantiert nicht, dass es mit dem 2D-Wert vergleichbar ist.

In [ ]:
RIDGE = ((428200, 5356500), (431200, 5356500))   # Talboden Kinzig -> Höhenrücken -> Harmersbachtal
ST_2D = ("tab:blue", 4.0, "-")                    # Linienstil: Farbe, Breite, Strich
ST_STRICH = (0, (4, 2))
PUFFER = 1500                                     # Suchfenster: Puffer um die Luftlinie in Metern
OBJ5 = Objective(weights={"cost": 1.0, "gradient": 5.0})
NUR_ERSTE_MELDUNG = "Path.total_cost is a recompute"

def route(paar, zeige_meldung=False, **kw):
    # eine Route mit Dijkstra; die bekannte total_cost-Meldung nur auf Wunsch zeigen
    with warnings.catch_warnings(record=True) as gesammelt:
        warnings.simplefilter("always")
        finder = PathFinder(RASTER_A, paar[0], paar[1], search_space_buffer_m=PUFFER,
                            neighborhood_str="r2", **kw)
        pfad = finder.find_route()
    for w in gesammelt:
        if NUR_ERSTE_MELDUNG in str(w.message):
            if zeige_meldung:
                print("pyorps sagt hier auf Englisch: total_cost ist eine Nachrechnung aus Zellwert × 2D-Länge. Sie berücksichtigt weder die 3D-Streckung noch die Neigung,")
                print("und die Zahl ist in Einheiten der Zielfunktion, nicht in EUR. Der Originaltext:")
                print(w.message)
        else:
            warnings.warn(w.message, w.category)
    return pfad

def hoehe_an(xy):
    # Höhe des DGM an Punkten (xy: n x 2), bilinear
    spalte = (xy[:, 0] - T.c) / T.a - 0.5
    zeile = (xy[:, 1] - T.f) / T.e - 0.5
    return map_coordinates(np.nan_to_num(dem, nan=np.nanmean(dem)), [zeile, spalte], order=1, mode="nearest")

def profil(pfad):
    # Koordinaten, zurückgelegte 2D-Strecke und Geländehöhe entlang der Route
    xy = np.asarray(pfad.path_coords)
    schritt = np.hypot(*np.diff(xy, axis=0).T)
    return xy, np.r_[0, np.cumsum(schritt)], hoehe_an(xy)

def kosten_auf_a(pfad):
    # Route auf Raster A bewerten: Zellen der Route im ganzen Raster, dann route_metrics (Zellwert x 2D-Länge)
    xy = np.asarray(pfad.path_coords)
    spalte = np.floor((xy[:, 0] - Tk.c) / Tk.a).astype(np.int64)
    zeile = np.floor((xy[:, 1] - Tk.f) / Tk.e).astype(np.int64)
    zellen = zeile * kosten.shape[1] + spalte
    zellen = zellen[np.r_[True, np.diff(zellen) != 0]]
    return route_metrics({0: zellen}, kosten, ZELLE)[0]["construction_cost"]

def kennzahlen(pfad):
    # gleiche Messung für 2D- und 3D-Routen (Höhen aus dem DGM entlang der Route)
    xy, lauf, h = profil(pfad)
    schritt, dh = np.diff(lauf), np.diff(h)
    return {"2D-Länge [m]": lauf[-1],
            "3D-Länge [m]": np.hypot(schritt, dh).sum(),
            "Kosten auf Raster A [EUR]": kosten_auf_a(pfad),
            "steilste Kante [%]": (np.abs(dh) / np.maximum(schritt, 1e-9) * 100).max(),
            "höchster Punkt [m]": h.max(),
            "Anstieg gesamt [m]": np.clip(dh, 0, None).sum()}

def tabelle(routen):
    return tab_de(pd.DataFrame({name: kennzahlen(p) for name, p in routen.items()}).T)

def karte(ax, routen, stile, rand=250, titel=""):
    # Schummerung im Fenster um alle Routen, Routen darüber
    alle = np.vstack([np.asarray(p.path_coords) for p in routen.values()])
    x0, y0 = alle.min(axis=0) - rand
    x1, y1 = alle.max(axis=0) + rand
    c0, r1 = zelle_von_xy(max(x0, EXT[0]), max(y0, EXT[2]))
    c1, r0 = zelle_von_xy(min(x1, EXT[1]), min(y1, EXT[3]))
    ext = (T.c + c0 * ZELLE, T.c + c1 * ZELLE, T.f - r1 * ZELLE, T.f - r0 * ZELLE)
    ax.imshow(hs[r0:r1, c0:c1], cmap="gray", extent=ext)
    ax.imshow(dem[r0:r1, c0:c1], cmap="terrain", alpha=0.35, extent=ext)
    for (name, p), (farbe, breite, strich) in zip(routen.items(), stile):
        xy = np.asarray(p.path_coords)
        ax.plot(xy[:, 0], xy[:, 1], color=farbe, lw=breite, ls=strich, label=name)
    ax.set_title(titel, fontsize=10)
    ax.legend(fontsize=8, loc="best")
    achsen(ax)

pfad_2d = route(RIDGE)                                                    # ohne DGM
pfad_3d = route(RIDGE, zeige_meldung=True, dem=DEM, objective=OBJ5)       # mit DGM, Gewicht 5
routen = {"2D (ohne DGM)": pfad_2d, "3D (Gewicht 5)": pfad_3d}

print(f"\npyorps meldet für die 3D-Route: total_length = {de(pfad_3d.total_length, 1)} m (3D), "
      f"total_length_2d = {de(pfad_3d.total_length_2d, 1)} m | ohne DGM ist total_length_3d = {pfad_2d.total_length_3d}")
assert np.isclose(kennzahlen(pfad_3d)["3D-Länge [m]"], pfad_3d.total_length_3d, rtol=1e-3)   # eigene Messung stimmt
eur_2d = kennzahlen(pfad_2d)["Kosten auf Raster A [EUR]"]
assert np.isclose(eur_2d, pfad_2d.total_cost, rtol=1e-3)
print(f"Prüfung: route_metrics bewertet die 2D-Route mit {de(eur_2d)} EUR, PathFinder meldet {de(pfad_2d.total_cost)} EUR.")
print(f"Zum Vergleich bei der 3D-Route: total_cost = {de(pfad_3d.total_cost)}, auf Raster A bewertet {de(kennzahlen(pfad_3d)['Kosten auf Raster A [EUR]'])} EUR.\n")
print(tabelle(routen).to_string())

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), gridspec_kw={"height_ratios": [2.2, 1]})
STILE_PAAR = [ST_2D, ("tab:red", 2.5, "-")]
karte(ax1, routen, STILE_PAAR, titel="Höhenrücken-Paar: Route ohne und mit DGM")
for p in RIDGE:
    ax1.plot(*p, "o", mfc="white", mec="black", ms=6)
for (name, p), (farbe, breite, strich) in zip(routen.items(), STILE_PAAR):
    _, lauf, h = profil(p)
    ax2.plot(lauf, h, color=farbe, lw=breite * 0.7, ls=strich, label=name)
ax2.set_xlabel("Strecke entlang der Route (2D) [m]", fontsize=8); ax2.set_ylabel("Höhe [m]", fontsize=8)
ax2.tick_params(labelsize=8)
ax2.set_title("Höhenprofil", fontsize=10); ax2.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Das Gewicht wählen

Das Gewicht legt fest, wie teuer ein Prozent Neigung ist. Merkregel: **Gewicht × Neigung in % sollte in der Größenordnung der Kosten pro Meter liegen**, sonst sieht die Suche die Neigung nicht (Gewicht zu klein) oder nur noch die Neigung (Gewicht zu groß).
Die Zelle rechnet diese Zahlen für Raster A aus und lässt dann die Gewichte 0, 2, 5 und 20 auf dem Höhenrücken-Paar laufen. In der Karte liegen Routen teils übereinander; deshalb sind einige Linien dick und andere gestrichelt darüber gezeichnet.

Danach machen wir dasselbe für das Hauptpaar `SOURCE` -> `TARGET` aus Teil 3. Ob sich die Route ändert, entscheiden die Zahlen, nicht unsere Erwartung.

In [ ]:
# Kosten pro Meter der 2D-Route gegen Gewicht × Neigung
GEWICHTE = (0, 2, 5, 20)
STILE_GEWICHT = [ST_2D, ("black", 1.6, ST_STRICH), ("tab:orange", 3.5, "-"), ("tab:red", 1.8, ST_STRICH), ("magenta", 2.4, "-")]

kosten_je_m = pfad_2d.total_cost / pfad_2d.total_length
print(f"Mittlere Kosten der 2D-Route: {de(kosten_je_m)} EUR pro Meter")
for w in GEWICHTE:
    print(f"Gewicht {w:>2}: bei 20 % Neigung kommen {de(w * 20):>3} pro Meter dazu ({de(100 * w * 20 / kosten_je_m)} % der Kosten)")

def gewichte_testen(paar, gewichte=GEWICHTE):
    # 2D-Route und je eine 3D-Route pro Gewicht
    routen = {"2D (ohne DGM)": route(paar)}
    for w in gewichte:
        routen[f"3D, Gewicht {w}"] = route(paar, dem=DEM,
                                           objective=Objective(weights={"cost": 1.0, "gradient": float(w)}))
    return routen

def abstand_m(pfad_a, pfad_b):
    # größter Abstand eines Punktes von Route b zur Route a (in Metern)
    punkte_a = np.asarray(pfad_a.path_coords)
    return cKDTree(punkte_a).query(np.asarray(pfad_b.path_coords))[0].max()

routen_ridge = gewichte_testen(RIDGE)
print()
print(tabelle(routen_ridge).to_string())

fig, ax = plt.subplots(figsize=(9, 5))
karte(ax, routen_ridge, STILE_GEWICHT, titel="Höhenrücken-Paar: Gewicht der Neigung 0, 2, 5, 20")
for p in RIDGE:
    ax.plot(*p, "o", mfc="white", mec="black", ms=6)
fig.tight_layout()
plt.show()

In [ ]:
# Hauptpaar aus Teil 3: gleiche Auswertung
routen_main = gewichte_testen((SOURCE, TARGET))
print(tabelle(routen_main).to_string())

namen = list(routen_main)
for name in namen[1:]:
    print(f"größter Abstand 2D-Route -> {name}: {de(abstand_m(routen_main[namen[0]], routen_main[name]))} m")
unterschied = max(abstand_m(routen_main[namen[0]], routen_main[n]) for n in namen[1:])
if unterschied > 100:
    print("\nErgebnis: Das Gelände ändert die Route für dieses Paar.")
else:
    print("\nErgebnis: Die Routen sind praktisch gleich. Das ist ein gültiges Ergebnis: das Gelände ändert nicht immer die Route.")
if abstand_m(routen_main[namen[0]], routen_main[namen[1]]) > 100:
    print("Schon bei Gewicht 0 ändert sich die Route: Die 3D-Streckung allein macht den steilen Hang teurer.")
w_aus = max(namen[1:], key=lambda n: abstand_m(routen_main[namen[1]], routen_main[n]))
print(f"Unterschied zwischen den 3D-Routen (Gewicht 0 gegen {w_aus}): {de(abstand_m(routen_main[namen[1]], routen_main[w_aus]))} m")

fig, ax = plt.subplots(figsize=(9, 5))
karte(ax, routen_main, STILE_GEWICHT, titel="Hauptpaar SOURCE -> TARGET: ohne und mit DGM")
ax.plot(*SOURCE, "o", mfc="white", mec="black", ms=6); ax.plot(*TARGET, "s", mfc="white", mec="black", ms=6)
fig.tight_layout()
plt.show()

## Harte Grenze und Delta-Stepping

**Harte Neigungsgrenze.** `GradientOptions(max_gradient_pct=...)` verbietet jeden Schritt, der steiler ist als der Grenzwert. Das ist kein Preis, sondern ein Verbot. Wird die Grenze zu klein, gibt es keine Route mehr, und pyorps wirft `NoPathFoundError`. Diesen Fehler fangen wir in der nächsten Zelle ab.
Gemessen wird die Neigung zwischen zwei benachbarten Zellen auf der Route (Kante), nicht die Neigung der Rasterzelle aus der Karte oben. Darum kann die Karte an einer Stelle mehr zeigen als der Grenzwert.

**Delta-Stepping mit DGM.** Delta-Stepping kann Neigungsterme nicht auswerten. Geben wir ein DGM, läuft deshalb Dijkstra, und pyorps sagt das mit einer `AlgorithmFallbackWarning`. Das ist kein Fehler, nur ein Hinweis, dass der gewünschte Algorithmus nicht läuft. Wir fangen die Meldung absichtlich ab, um sie zu lesen.

In [ ]:
# Harte Neigungsgrenze auf dem Höhenrücken-Paar (nur Kosten, dazu das Limit)
def route_mit_grenze(paar, grenze):
    return route(paar, dem=DEM,
                 objective=Objective(weights={"cost": 1.0},
                                     gradient_options=GradientOptions(max_gradient_pct=float(grenze))))

routen_grenze = {"2D (ohne DGM)": pfad_2d, "3D, ohne Grenze": route(RIDGE, dem=DEM, objective=Objective(weights={"cost": 1.0}))}
for grenze in (40, 30, 20, 10):
    routen_grenze[f"Grenze {grenze} %"] = route_mit_grenze(RIDGE, grenze)
tab = tabelle(routen_grenze)
print(tab.to_string())

# eine zu kleine Grenze: keine Route
try:
    route_mit_grenze(RIDGE, 5)
except NoPathFoundError as fehler:
    print("\nGrenze 5 %: Es gibt keine Route mehr. pyorps meldet das als NoPathFoundError, der Text ist englisch")
    print("(die beiden Zahlen darin sind interne Nummern, keine Koordinaten):")
    print(str(fehler)[:150], "...")
else:
    print("\nGrenze 5 %: unerwartet doch eine Route")

fig, ax = plt.subplots(figsize=(9, 5))
STILE_GRENZE = [ST_2D, ("black", 1.6, ST_STRICH), ("tab:orange", 3.5, "-"), ("tab:red", 1.8, ST_STRICH), ("magenta", 2.4, "-"), ("darkgreen", 2.4, "-")]
karte(ax, routen_grenze, STILE_GRENZE, titel="Höhenrücken-Paar: harte Neigungsgrenze")
for p in RIDGE:
    ax.plot(*p, "o", mfc="white", mec="black", ms=6)
fig.tight_layout()
plt.show()

In [ ]:
# Delta-Stepping mit DGM: pyorps fällt auf Dijkstra zurück und warnt
with warnings.catch_warnings(record=True) as gesammelt:
    warnings.simplefilter("always")
    finder = PathFinder(RASTER_A, RIDGE[0], RIDGE[1], search_space_buffer_m=PUFFER,
                        neighborhood_str="r2", dem=DEM, objective=OBJ5)
    pfad_ds = finder.find_route(algorithm="delta-stepping")

print("pyorps warnt auf Englisch: Delta-Stepping kann Neigungsterme nicht auswerten, deshalb läuft Dijkstra. Der Originaltext:")
for w in gesammelt:
    if issubclass(w.category, AlgorithmFallbackWarning):
        print(w.category.__name__, "->", w.message)
print("path.algorithm mit DGM:", pfad_ds.algorithm, "(gewünscht war delta-stepping)")
print("gleiche Länge wie die Dijkstra-Route von oben:", np.isclose(pfad_ds.total_length, pfad_3d.total_length))

# zum Vergleich ohne DGM: Delta-Stepping läuft wirklich
pfad_ds2 = PathFinder(RASTER_A, RIDGE[0], RIDGE[1], search_space_buffer_m=PUFFER,
                      neighborhood_str="r2").find_route(algorithm="delta-stepping")
print("path.algorithm ohne DGM:", pfad_ds2.algorithm)

In [ ]:
# 3D-Bild: Gelände im Fenster der Routen, Routen als Linien auf Geländehöhe
STUFE = 4                                   # jedes 4. Rasterelement (20 m) für ein schnelles Bild
alle = np.vstack([np.asarray(p.path_coords) for p in (pfad_2d, pfad_3d)])
x0, y0 = alle.min(axis=0) - 250
x1, y1 = alle.max(axis=0) + 250
c0, r1 = zelle_von_xy(max(x0, EXT[0]), max(y0, EXT[2]))
c1, r0 = zelle_von_xy(min(x1, EXT[1]), min(y1, EXT[3]))
Z = dem[r0:r1:STUFE, c0:c1:STUFE]
xs = T.c + (np.arange(c0, c1, STUFE) + 0.5) * ZELLE - x0          # Meter ab der Fensterecke
ys = T.f + (np.arange(r0, r1, STUFE) + 0.5) * T.e - y0
X, Y = np.meshgrid(xs, ys)
farben = LightSource(315, 45).shade(Z, plt.cm.terrain, vert_exag=1, blend_mode="soft", dx=STUFE * ZELLE, dy=STUFE * ZELLE)

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(projection="3d", computed_zorder=False)       # feste Zeichenreihenfolge: erst Fläche, dann Linien
ax.plot_surface(X, Y, Z, facecolors=farben, rstride=1, cstride=1, linewidth=0, antialiased=False, shade=False, zorder=1)
for p, farbe, name in ((pfad_2d, "tab:blue", "2D-Route"), (pfad_3d, "tab:red", "3D-Route (Gewicht 5)")):
    xy, _, h = profil(p)
    ax.plot(xy[:, 0] - x0, xy[:, 1] - y0, h + 8, color=farbe, lw=2.2, label=name, zorder=10)   # 8 m über dem Boden gezeichnet
ax.set_box_aspect((x1 - x0, y1 - y0, 2.0 * (np.nanmax(Z) - np.nanmin(Z))), zoom=1.15)   # Höhe 2-fach überhöht
ax.view_init(elev=35, azim=-80)
ax.set_xlabel("Ost [m ab Fensterecke]", fontsize=8, labelpad=6)
ax.tick_params(labelsize=8)
ax.legend(fontsize=8, loc="upper left")
# Die Namen der y- und z-Achse stehen im Titel: Achsenbeschriftungen von 3D-Bildern werden im Notebook oft abgeschnitten
ax.set_title("Gelände (2-fach überhöht) mit 2D- und 3D-Route\n"
             "x: Ost, y: Nord (Meter ab Fensterecke), z: Höhe (m ü. NHN)", fontsize=9)
fig.subplots_adjust(left=0.02, right=0.86, bottom=0.04, top=0.94)
plt.show()

## Was wir mitnehmen

**Länge und Kosten mit DGM**
- `total_length` ist mit DGM die **3D-Länge**. Dazu gibt es `total_length_2d`, `total_length_3d`, `mean_gradient_pct` und `max_gradient_pct`. Ohne DGM sind diese vier `None` und `total_length` ist die 2D-Länge. `path_geometry.length` ist immer die 2D-Länge der Linie.
- `total_cost` bleibt auch mit DGM die **2D-Nachrechnung** auf der Suchfläche (Zellwert × 2D-Länge) und ist bei 3D-Routen laut pyorps in Einheiten der Zielfunktion angegeben, **nicht in EUR garantiert** (siehe die Meldung oben; die Zahlen im Vergleich stehen in der Zelle oben). Das hat die Suche auch nicht minimiert: Gesucht wurde `feasibility` (Kosten plus Gewicht × Neigung über 3D-Längen); die Teile stehen in `Path.metrics`. pyorps warnt davor einmal pro `PathFinder` mit einer `UserWarning` und schreibt es in `Path.total_cost_basis`. Um Routen fair zu vergleichen, bewerten wir sie wie in diesem Teil mit `route_metrics` auf einem Raster.
- Die 3D-Streckung `sqrt(1 + Neigung²)` (Neigung als Verhältnis) gilt nach dem Quelltext immer, sobald ein DGM da ist.

**Gewicht und Grenze**
- Gewicht × Neigung in % soll zu den Kosten pro Meter passen (Zelle oben). Das Gewicht macht Neigung teurer, die Grenze verbietet Schritte. Eine Grenze kann jede Route unmöglich machen (`NoPathFoundError`).
- Ohne `objective=` rechnet pyorps nach dem Quelltext mit einer eingebauten exponentiellen Neigungsstrafe (hier nicht ausprobiert). Wir nehmen immer ein eigenes `Objective`, dann ist klar, was gerechnet wird.

**Abdeckung des DGM** (aus der pyorps-Dokumentation und dem Quelltext von Version 0.5.0; in diesem Notebook rechnen wir das nicht nach)
- Zellen, die das DGM nicht abdeckt (außerhalb, Nodata, Löcher), sind **gesperrt**, ohne Warnung. Nur wenn ein Start- oder Zielpunkt dort liegt und verschoben wird, meldet pyorps eine `CoverageWarning`.
- Das DGM muss im gleichen Koordinatensystem sein wie das Kostenraster (hier EPSG:25832). pyorps rechnet nicht um und bricht mit `InvalidInputError` ab. Ein DGM ohne Koordinatensystem gilt als im System des Rasters.
- Das DGM wird **bilinear** auf das Suchfenster des Kostenrasters umgerechnet. Am Fensterrand kann ein schmaler Streifen ohne Höhe bleiben; ein großzügiger Puffer hilft.
- Als Datei oder als Tupel `(feld, crs, transform)`. Ein nacktes Array geht bei einem Kostenraster aus einer Datei nicht.

**Auflösung.** Ein gröberes DGM (z. B. 25 m statt 5 m) ist schneller zu laden; schmale steile Teile werden beim Mitteln flacher. Ob die Route sich ändert, probiert ihr in Aufgabe 3 aus.

**Delta-Stepping** läuft mit DGM nicht (`AlgorithmFallbackWarning`, `path.algorithm == "dijkstra"`).

**GUI.** Die Karte „Objective & terrain" in der GUI (Teil 7) stellt dasselbe ein: DGM-Datei, Gewichte, Neigungsgrenze. Die GUI verweigert die Kombination DGM + Delta-Stepping und zeigt eine Meldung am Feld; die Bibliothek fällt dagegen mit Warnung auf Dijkstra zurück.

### Übung Teil 5

Arbeitet mit neuen Variablennamen; `RASTER_A`, `DEM`, `SOURCE` und `TARGET` bleiben unverändert.

1. **Zweites Paar.** Rechnet mit `gewichte_testen(((428663, 5356873), (431273, 5356048)))` und `tabelle(...)` das Paar `SOURCES[2]` -> `TARGETS[3]` aus Teil 3. Ändern sich Länge und steilste Kante mit dem Gewicht? Zeichnet die Routen mit `karte(ax, routen, STILE_GEWICHT)`.
2. **Kleinste Grenze.** Sucht für das Hauptpaar `SOURCE` -> `TARGET` mit `route_mit_grenze` die kleinste ganzzahlige Neigungsgrenze, bei der noch eine Route existiert (Tipp: fangt `NoPathFoundError` ab). Wie ändert sich die Länge der Route an dieser Grenze?
3. **Gröberes DGM.** Mittelt das DGM auf 25 m: `dem25 = dem.reshape(224, 5, 224, 5).mean(axis=(1, 3)).astype("float32")`, dazu `from affine import Affine` und `transform25 = Affine(25, 0, T.c, 0, -25, T.f)`. Übergebt `dem=(dem25, "EPSG:25832", transform25)` an `PathFinder` mit dem Gewicht 5 auf dem Höhenrücken-Paar. Vergleicht `total_length` und `max_gradient_pct` mit der 5-m-Route `pfad_3d` aus der Zelle oben.

---
# Teil 6: Freileitungen mit `ConstrainedPathFinder`

Ein Kabel liegt im Boden und folgt jeder Zelle seiner Route. Eine **Freileitung** hängt in der Luft: Der Leiter läuft gerade von Mast zu Mast. Daraus folgen neue Regeln:

- **Masten:** Sie kosten viel (Fundament und Stahl) und stehen nur auf erlaubten Flächen.
- **Spannfeld:** Das ist das Stück Leitung zwischen zwei Masten. Seine Länge heißt **Spannweite**; sie liegt zwischen einem kleinsten und einem größten Wert.
- **Winkel:** Die Leitung knickt nur an einem Mast. Je schärfer der Knick, desto schwerer und teurer der Mast.
- **Durchhang und Bodenabstand:** Der Leiter hängt zwischen den Masten durch. An keiner Stelle darf er dem Boden (oder Bäumen) näher kommen als ein Mindestabstand.

**Was `ConstrainedPathFinder` macht:** Er sucht die Route und die Maststandorte in *einer* Suche. Ein Zustand der Suche ist nicht nur die Zelle, sondern das Tripel (Zelle, Richtung, Länge seit dem letzten Mast). So kann die Suche Winkel- und Spannweitengrenzen schon während der Suche einhalten, statt erst eine Route zu suchen und dann Masten daraufzusetzen.

**Masttypen** (die Namen kommen aus dem Profil und stehen so im Ergebnis):

| im Ergebnis | auf Deutsch | wann |
|---|---|---|
| `terminal` | Endmast | am Start- und am Zielpunkt |
| `suspension` | Tragmast | Knick bis zur ersten Grenze des Profils (110 kV: 8°) |
| `light_angle` | leichter Winkelmast | größerer Knick (110 kV: bis 20°) |
| `heavy_angle` | schwerer Winkelmast | 110 kV: bis 30° |
| `strain` | Abspannmast | 110 kV: bis 40° |

**Grenzen der Suche** (aus dem Code von pyorps 0.5.0; die ersten beiden zeigt die kleine Testzelle in 6.2):
- Verbotene Zellen (Wert 65535) sperren auch den Leiter, nicht nur den Maststandort: Eine Wand aus einer einzigen verbotenen Zelle trennt Start und Ziel. Die Leitung kann also keine verbotene Fläche überspannen.
- Findet die Suche keine Leitung, gibt sie ein **leeres Ergebnis ohne Warnung** zurück (0 Masten). Unsere Hilfsfunktion `fl_suche` prüft das deshalb nach jeder Suche.
- An jeder Richtungsänderung steht ein Mast. Jedes Spannfeld ist deshalb eine gerade Linie im Raster.
- Geländemodell (DGM) und Oberflächenmodell (DOM) passt `ConstrainedPathFinder` nach der **Form** des Arrays an das Suchfenster an, nicht nach Koordinaten. Wir geben beide deshalb genau auf dem Gitter des Kostenrasters ein (Setup-Zelle).

In [ ]:
# Setup für Teil 6: Importe, Variablen aus Teil 3, DGM auf dem Gitter des Kostenrasters
import dataclasses
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LightSource
from rasterio.features import rasterize

from pyorps import PathFinder, InvalidInputError, full_window_buffer_m
# Diese beiden Klassen exportiert pyorps 0.5.0 nicht auf oberster Ebene, deshalb die langen Importpfade:
from pyorps.core.infrastructure_profile import InfrastructureProfile
from pyorps.graph.constrained_path_finder import ConstrainedPathFinder


def de(x, nd=0):
    """Zahl im deutschen Format: Punkt für Tausender, Komma für Dezimalstellen."""
    return f"{x:,.{nd}f}".replace(",", "§").replace(".", ",").replace("§", ".")


# (falls Teil 3 nicht ausgeführt wurde): dieselben Werte wie dort
if "RASTER_A" not in globals():
    BBOX = (427000, 5354000, 432000, 5359000)
    CRS = "EPSG:25832"
    RASTER_A = "data/cost_raster_a.tiff"
    SOURCE = (427558, 5357403)                   # Kinzigtal-Talboden
    TARGET = (430288, 5355723)                   # westliches Ende des Harmersbachtals

FL_PROFILE = Path("data/profiles")              # Kopien der Profile aus dem pyorps-Repository
FL_DGM_DATEI = hole_dgm() if "hole_dgm" in globals() else "data/dem_clip.tif"   # hole_dgm() aus Teil 5

with rasterio.open(RASTER_A) as src:
    fl_kosten, FL_T, FL_CRS = src.read(1), src.transform, src.crs
with rasterio.open(FL_DGM_DATEI) as src:
    _dgm, _dt, _nodata = src.read(1).astype("float32"), src.transform, src.nodata

# Das DGM hat 300 m Rand um das Gebiet. Wir schneiden es auf das Gitter des Kostenrasters zu:
# gleiche Zellgröße, Versatz in ganzen Zellen, danach gleiche Form.
assert np.isclose(_dt.a, FL_T.a) and np.isclose(_dt.e, FL_T.e)
_spalte0, _zeile0 = (FL_T.c - _dt.c) / FL_T.a, (_dt.f - FL_T.f) / abs(FL_T.e)
assert float(_spalte0).is_integer() and float(_zeile0).is_integer()
_spalte0, _zeile0 = int(_spalte0), int(_zeile0)
fl_dgm = _dgm[_zeile0:_zeile0 + fl_kosten.shape[0], _spalte0:_spalte0 + fl_kosten.shape[1]]
assert fl_dgm.shape == fl_kosten.shape and not np.any(fl_dgm == _nodata)
FL_DGM = (fl_dgm, FL_CRS, FL_T)                 # so bekommt ConstrainedPathFinder das DGM: (Array, CRS, Transform)
FL_PUFFER = full_window_buffer_m(RASTER_A)      # Suchfenster = ganzes Raster (warum: siehe 6.2)
FL_TAL = ((427558, 5357403), (427558, 5356400))  # kurzes Punktepaar im Talboden (1 km nach Süden), für 6.6 und die Übung
FL_EXT = (FL_T.c, FL_T.c + FL_T.a * fl_kosten.shape[1], FL_T.f + FL_T.e * fl_kosten.shape[0], FL_T.f)
print(f"DGM: {FL_DGM_DATEI} | zugeschnitten ab Zeile {_zeile0}, Spalte {_spalte0} | Form {fl_dgm.shape} | "
      f"Höhe {de(fl_dgm.min())} bis {de(fl_dgm.max())} m")

FL_TYPEN = {"terminal": ("Endmast", "black"), "suspension": ("Tragmast", "tab:green"),
            "light_angle": ("leichter Winkelmast", "tab:orange"), "heavy_angle": ("schwerer Winkelmast", "tab:red"),
            "strain": ("Abspannmast", "tab:purple")}


def fl_lade_profil(kv, regel="remove_if_valid"):
    """Profil laden; die Endfeldregel setzen wir hier auf "remove_if_valid" (in den Dateien steht "remove", siehe 6.6)."""
    profil = InfrastructureProfile.load(str(FL_PROFILE / f"overhead_line_{kv}kv.yaml"))
    return dataclasses.replace(profil, terminal_span_rule=regel)


def fl_suche(profil, quelle=None, ziel=None, puffer=None, nachbarschaft="r2", **kw):
    """Eine Freileitungssuche; bricht bei leerem Ergebnis ab."""
    quelle, ziel = quelle or SOURCE, ziel or TARGET
    t0 = time.perf_counter()
    with warnings.catch_warnings(record=True) as meldungen:
        warnings.simplefilter("always")
        cpf = ConstrainedPathFinder(RASTER_A, quelle, ziel, profile=profil, neighborhood_str=nachbarschaft,
                                    search_space_buffer_m=puffer or FL_PUFFER, **kw)
        erg = cpf.find_route()
    for m in meldungen:
        if "CUDA" not in str(m.message):
            print(f"  Meldung von pyorps (englisch, {m.category.__name__}): {str(m.message)[:420]}")
    if erg.n_towers == 0:
        raise RuntimeError("Leeres Ergebnis: keine Freileitung im Suchfenster gefunden. "
                           "Größeres Fenster (search_space_buffer_m) oder anderes Punktepaar wählen.")
    print(f"  {profil.name}: {erg.n_towers} Masten, Suche {de(time.perf_counter() - t0, 1)} s")
    return erg, cpf


def fl_tabelle(ergebnisse):
    """Eine Zeile pro Leitung, kurze Spalten (ganze Zahlen, ohne Tausenderzeichen)."""
    zeilen = {}
    for name, erg in ergebnisse.items():
        maste = erg.towers_to_geodataframe(crs=FL_CRS)
        hoehen = maste.loc[maste["tower_type"] != "terminal", "height_m"].value_counts().sort_index()
        zeilen[name] = {"Masten": erg.n_towers, "Länge [m]": round(erg.total_length),
                        "Spannweite [m]": f"{erg.min_span_actual_m:.0f}-{erg.max_span_actual_m:.0f}",
                        "Höhe [m] × Anzahl": ", ".join(f"{h:g}×{n}" for h, n in hoehen.items()),
                        "Raster [EUR]": round(erg.total_terrain_cost), "Masten [EUR]": round(erg.total_tower_cost),
                        "Knicke [EUR]": round(erg.total_angle_penalty_cost), "Gesamt [EUR]": round(erg.total_cost)}
    return pd.DataFrame(zeilen).T


def fl_typen(ergebnisse):
    """Anzahl der Masten je Typ, eine Zeile pro Leitung."""
    return pd.DataFrame({name: {FL_TYPEN[t][0]: erg.tower_type_counts.get(t, 0) for t in FL_TYPEN}
                         for name, erg in ergebnisse.items()}).T


FL_SPALTEN = ("Spalten: Masten mit den 2 Endmasten; Höhe = Aufhängehöhe der Linienmasten; Raster = Rasterkosten der "
              "Route (Wert × Meter); Masten = Fundamente, Mast-Grundpreise und Höhenaufpreise; Knicke = Zuschläge für Knicke")


def fl_karte(ax, ergebnisse, farben, hintergrund="kosten", rand=300, maste=True):
    """Routen (und Masten nach Typ) auf Kostenraster oder Schummerung."""
    if hintergrund == "kosten":
        bild = np.ma.masked_equal(fl_kosten, 65535)
        cmap = plt.get_cmap("YlGnBu").copy(); cmap.set_bad("0.25")
        bild = ax.imshow(bild, cmap=cmap, extent=FL_EXT, vmin=50, vmax=700)
    else:
        bild = ax.imshow(LightSource(azdeg=315, altdeg=45).hillshade(fl_dgm, vert_exag=1, dx=5, dy=5),
                         cmap="gray", extent=FL_EXT)
    alle = []
    for (name, erg), farbe in zip(ergebnisse.items(), farben):
        xy = np.asarray(erg.path_coords); alle.append(xy)
        ax.plot(xy[:, 0], xy[:, 1], color=farbe, lw=1.8, label=name,
                path_effects=[pe.Stroke(linewidth=3.6, foreground="white"), pe.Normal()])
        if maste and hasattr(erg, "towers"):
            for t in erg.towers:
                ax.plot(t.location.x, t.location.y, "o", ms=5, mfc=FL_TYPEN[t.tower_type][1], mec="white", mew=0.6)
    alle = np.vstack(alle)
    ax.set_xlim(alle[:, 0].min() - rand, alle[:, 0].max() + rand)
    ax.set_ylim(alle[:, 1].min() - rand, alle[:, 1].max() + rand)
    ax.set_aspect("equal"); ax.tick_params(labelsize=8)
    ax.ticklabel_format(useOffset=False, style="plain")
    ax.set_xlabel("Rechtswert [m]", fontsize=8); ax.set_ylabel("Hochwert [m]", fontsize=8)
    return bild


def fl_legende(ax, typen=True, ncol=4):
    """Legende unter der Karte, damit sie keine Route verdeckt."""
    if typen:
        for typname, farbe in FL_TYPEN.values():
            ax.plot([], [], "o", mfc=farbe, mec="white", label=typname)
    ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=ncol, frameon=False)

## 6.1 Das Profil: alle Regeln einer Leitung in einer Datei

Ein `InfrastructureProfile` beschreibt eine Leitungsart. pyorps bringt im Repository drei Profile für 110, 220 und 380 kV mit (Ordner `profiles/`). Sie sind **nicht** im installierten Paket (pip/wheel) enthalten, deshalb liegen Kopien unter `data/profiles/`.

Die Felder, die wir in diesem Teil benutzen (Bedeutung nach dem Code von pyorps 0.5.0):

| Feld | Bedeutung |
|---|---|
| `hard_angle_limit_deg` | größte Richtungsänderung zwischen zwei Rasterschritten; schärfere Knicke sind verboten |
| `min_span_m`, `max_span_m` | kürzeste und längste Spannweite zwischen zwei Masten |
| `tower_cost_params["angle_types"]` | Masttyp nach Knickwinkel am Mast, mit Mast-Grundpreis |
| `tower_cost_params["terrain_cost_map"]` | Fundamentpreis in EUR pro m² je Rasterwert (linear interpoliert), mal `tower_ground_area_m2` |
| `angle_cost_function`, `angle_cost_params` | Zuschlag pro Knick nach Winkel (`piecewise`: linear zwischen den Stützstellen) |
| `max_gradient_percent`, `gradient_cost_params["scale"]` | nur mit DGM: steilster erlaubter Rasterschritt; Faktor `exp(scale × s²)` pro Schritt (s = Steigung als Verhältnis) |
| `tower_heights_m` | Aufhängehöhen des Leiters am Mast (nicht die Mastspitze), eine pro Höhenklasse |
| `conductor_weight_per_m`, `conductor_tension_n`, `min_clearance_m` | Leitergewicht (N/m), Zugkraft (N), Mindestabstand zu Boden und Hindernis (m) |
| `terminal_span_rule` | was mit einem Mast passiert, der näher als `min_span_m` an einem Endpunkt steht (6.6) |

Weitere Felder, kurz: `soft_angle_limit_deg` muss kleiner gleich der harten Grenze sein, die Suche liest es sonst nicht. `span_bin_size_m` ist Pflicht, die Suche fasst Spannweiten aber in Klassen der Breite `min_span_m` zusammen. `tower_height_cost_per_increment` ist der Aufpreis je 3 m über der niedrigsten Höhe. `terminal_tower_height_m` und `terminal_tower_cost` (Endmast) stehen nur im Bericht. `tower_cost_function`, `gradient_cost_function` und `recommended_buffer_m` liest die Suche nicht.

Die Bodenabstandsprüfung läuft nur, wenn ein DGM da ist **und** das Profil Masthöhe, Leitergewicht, Zugkraft und Mindestabstand hat.

In [ ]:
# Profil laden und die Felder zeigen, die wir benutzen
fl_p110 = fl_lade_profil(110)
fl_datei_110 = InfrastructureProfile.load(str(FL_PROFILE / "overhead_line_110kv.yaml"))   # unverändert
fl_werte = {f: getattr(fl_p110, f) for f in ("hard_angle_limit_deg", "min_span_m", "max_span_m", "max_gradient_percent",
                                             "tower_heights_m", "conductor_weight_per_m", "conductor_tension_n",
                                             "min_clearance_m")}
print(pd.Series({k: (", ".join(f"{h:g}" for h in v) if isinstance(v, list) else f"{v:g}") for k, v in fl_werte.items()},
                name=fl_p110.name).to_string())
print(f"terminal_span_rule        {fl_p110.terminal_span_rule} (hier überschrieben, "
      f"in der Datei steht {fl_datei_110.terminal_span_rule})")

print("\nMasttypen (Knick bis ... Grad, Mast-Grundpreis):")
print(pd.DataFrame({FL_TYPEN[k][0]: {"Knick bis [°]": int(v["max_angle_deg"]), "Mast-Grundpreis [EUR]": int(v["base_cost"])}
                    for k, v in fl_p110.tower_cost_params["angle_types"].items()}).T.to_string())

# Fundamentpreis je Mast für einige Werte von Kostenraster A (Tabelle × Fundamentfläche) und Höhenaufpreise
fl_fundament = fl_p110.precompute_tower_terrain_costs()
print("\nFundament pro Mast [EUR] nach Rasterwert (365 = Wald):")
print(pd.Series({w: round(float(fl_fundament[w])) for w in (74, 200, 250, 365, 712)}).to_string())
fl_aufpreis = dict(zip(fl_p110.effective_tower_heights_m, fl_p110.precompute_height_premium()))
print("Höhenaufpreis je Mast [EUR]:", ", ".join(f"{h:g} m: {de(float(fl_aufpreis[h]))}" for h in sorted(fl_aufpreis)))

# Ein eigenes Profil im Code (Pflicht: Name, Beschreibung, beide Winkelgrenzen, Winkelkostenform)
fl_eigen = InfrastructureProfile(
    name="mein_110kv_test", description="kleines Beispiel",
    soft_angle_limit_deg=5.0, hard_angle_limit_deg=30.0,
    angle_cost_function="linear", angle_cost_params={"scale": 1000.0},   # 1000 EUR pro Grad Knick
    min_span_m=60, max_span_m=250, span_bin_size_m=10,
    tower_cost_params={"terrain_cost_map": {"0": 300, "500": 1500},
                       "angle_types": {"suspension": {"max_angle_deg": 5, "base_cost": 30000},
                                       "strain": {"max_angle_deg": 30, "base_cost": 200000}}},
    tower_ground_area_m2=64.0)
fl_kurz = dataclasses.replace(fl_p110, name="110kv_kurze_felder", max_span_m=200)   # Kopie mit einer Änderung
print(f"\nEigenes Profil: {fl_eigen.name} | Spannweitengrenzen: {fl_eigen.has_span_constraints} | "
      f"Bodenabstand prüfbar: {fl_eigen.has_clearance_constraints}")
print(f"Kopie: {fl_kurz.name}, max_span_m = {fl_kurz.max_span_m} | Original: {fl_p110.max_span_m}")

## 6.2 Die erste Freileitung (noch ohne Gelände)

Wir verbinden `SOURCE` und `TARGET` aus Teil 3 auf Kostenraster A mit dem 110-kV-Profil.

**Suchfenster:** Die kleine Testzelle unten zeigt: Mit einem Puffer von 200 m oder 400 m um die Luftlinie findet die Suche **keine** Leitung (0 Masten, ohne Warnung), obwohl `PathFinder` im selben Fenster ein Kabel findet. Die zweite Hälfte der Zelle zeigt den Grund an einem Spielraster: Eine Wand aus einer einzigen verbotenen Zelle sperrt die Leitung; mit einer Lücke geht die Route durch die Lücke. Im Kostenraster seht ihr in der Karte weiter unten schmale verbotene Streifen (grau), zum Beispiel entlang der Bahn. Die gefundene Leitung nimmt deshalb einen weiten Umweg nach Westen und Süden, der in den kleinen Fenstern nicht liegt.

Wir suchen also im ganzen Raster (`full_window_buffer_m`): 1 Million Zellen × 16 Richtungen × 6 Spannweitenklassen. In unserem Lauf belegte die Suche ohne Gelände etwa 1,5 GB Arbeitsspeicher.

**Nachbarschaft:** `r2` (16 Richtungen). Mit `r1` (8 Richtungen, 45° Abstand) wäre bei 40° harter Grenze jeder Knick verboten.

In [ ]:
# Kleine Testläufe (je unter einer Sekunde): zu kleines Fenster und eine Sperrwand
with warnings.catch_warnings(record=True) as fl_meldungen:
    warnings.simplefilter("always")
    for fl_puffer in (200, 400):
        fl_klein = ConstrainedPathFinder(RASTER_A, SOURCE, TARGET, profile=fl_p110,
                                         search_space_buffer_m=fl_puffer).find_route()
        fl_kabel_klein = PathFinder(RASTER_A, SOURCE, TARGET, search_space_buffer_m=fl_puffer,
                                    neighborhood_str="r2").find_route()
        print(f"Puffer {fl_puffer} m: Freileitung {fl_klein.n_towers} Masten, "
              f"Kabel (PathFinder) {de(fl_kabel_klein.total_length)} m")
    # Spielraster 1 km × 1 km, überall Wert 100, in Spalte 100 eine Wand aus EINER verbotenen Zelle
    fl_wand = np.full((200, 200), 100, np.uint16)
    fl_wand[:, 100] = 65535
    fl_wand_t = rasterio.transform.from_origin(500000, 5600000, 5, 5)
    fl_a, fl_b = (500102.5, 5599497.5), (500902.5, 5599497.5)      # Spalte 20 und 180, Zeile 100
    for fl_titel in ("Wand ohne Lücke", "Wand mit Lücke im Süden"):
        if fl_titel.endswith("Süden"):
            fl_wand[180:, 100] = 100                                 # die untersten 20 Zellen frei
        fl_r = ConstrainedPathFinder(fl_wand, fl_a, fl_b, profile=fl_p110, transform=fl_wand_t, crs="EPSG:25832",
                                     search_space_buffer_m=2000).find_route()
        fl_text = (f", südlichster Punkt der Route {de(fl_b[1] - np.asarray(fl_r.path_coords)[:, 1].min())} m "
                   f"südlich der Luftlinie" if fl_r.n_towers else "")
        print(f"{fl_titel}: {fl_r.n_towers} Masten{fl_text}")
fl_andere = [str(m.message)[:100] for m in fl_meldungen if "CUDA" not in str(m.message)]
print("Warnungen bei diesen Läufen:", fl_andere or "keine")

In [ ]:
# Freileitung 110 kV ohne Gelände im ganzen Raster, dazu das Kabel (PathFinder) im selben Fenster
fl_flach, fl_cpf = fl_suche(fl_p110)
fl_kabel = PathFinder(RASTER_A, SOURCE, TARGET, search_space_buffer_m=FL_PUFFER, neighborhood_str="r2").find_route()

fl_maste = fl_flach.towers_to_geodataframe(crs=FL_CRS)
assert np.isclose(fl_maste["total_cost"].sum(), fl_flach.total_tower_cost)   # Mastkosten = Summe der Masten
fl_maste_tab = pd.DataFrame({   # Zahlen mit Dezimalkomma, ohne Tausenderzeichen
    "Typ": fl_maste["tower_type"].map(lambda t: FL_TYPEN[t][0]),
    "Knick [°]": fl_maste["turn_angle_deg"].map(lambda v: de(v, 1)),
    "Spannweite davor [m]": fl_maste["span_to_previous_m"].map(lambda v: "-" if pd.isna(v) else de(v, 1)),
    "Höhe [m]": fl_maste["height_m"].map(lambda v: f"{v:g}"),
    "Fundament [EUR]": fl_maste["terrain_cost"].round().astype(int),
    "Mast-Grundpreis [EUR]": fl_maste["angle_cost"].round().astype(int),
    "Mast gesamt [EUR]": fl_maste["total_cost"].round().astype(int)})
print(fl_maste_tab.head(8).to_string())
print(f"... insgesamt {len(fl_maste)} Masten")

fl_vergleich = pd.DataFrame({
    "Kabel (PathFinder)": {"Länge [m]": round(fl_kabel.total_length), "Raster [EUR]": round(fl_kabel.total_cost),
                           "Masten [EUR]": 0, "Gesamt [EUR]": round(fl_kabel.total_cost)},
    "Freileitung 110 kV": {"Länge [m]": round(fl_flach.total_length), "Raster [EUR]": round(fl_flach.total_terrain_cost),
                           "Masten [EUR]": round(fl_flach.total_tower_cost), "Gesamt [EUR]": round(fl_flach.total_cost)}}).T
print(); print(fl_vergleich.to_string())
print(f"\nAnteil der Masten an den Gesamtkosten der Freileitung: {de(100 * fl_flach.total_tower_cost / fl_flach.total_cost)} %")

# Welche Knicke kann das r2-Raster überhaupt? (Richtungen der 16 Schritte)
fl_schritte = np.asarray(fl_cpf.steps, dtype=float)
fl_richtung = np.degrees(np.arctan2(fl_schritte[:, 0], fl_schritte[:, 1]))
fl_knicke = sorted({round(abs((a - b + 180) % 360 - 180), 1) for a in fl_richtung for b in fl_richtung} - {0.0})
print("kleinste mögliche Knicke in r2:", ", ".join(f"{de(k, 1)}°" for k in fl_knicke[:3]))

fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
fl_bild = fl_karte(ax, {"Freileitung 110 kV": fl_flach, "Kabel (PathFinder)": fl_kabel}, ["crimson", "black"])
fig.colorbar(fl_bild, ax=ax, shrink=0.7, label="Kosten [EUR/m]")
fl_legende(ax)
ax.set_title("Freileitung und Kabel auf Kostenraster A (grau = verboten)", fontsize=9)
plt.show()

**Was wir sehen:**
- Die Freileitung ist länger als das Kabel. Sie kann sich nicht durch die engen Lücken im Talboden schlängeln, denn jeder Knick kostet einen Mast und zwischen zwei Masten liegen mindestens 50 m gerade Strecke.
- Die Masten machen den größten Teil der Kosten aus; die Rasterkosten (Rasterwert × Meter, wie beim Kabel) sind der kleinere Teil.
- Im Raster `r2` ist der kleinste Knick 18,4°. Ein Tragmast (bis 8°) steht deshalb nur auf geraden Stücken; an jedem Knick steht ein Winkel- oder Abspannmast.
- Die Kosten des Kabels und der Freileitung sind hier nur Rasterpreise. Ein echter Vergleich Kabel gegen Freileitung braucht auch die Kabelkosten pro Meter.

## 6.3 Drei Spannungsebenen

Dasselbe Punktepaar mit den Profilen für 110, 220 und 380 kV. Die Profile unterscheiden sich vor allem in der längsten Spannweite (300, 350, 400 m), in der Fundamentfläche (100, 144, 256 m²), in den Mastpreisen und in der harten Winkelgrenze (40°, 35°, 30°).

In [ ]:
# Drei Spannungsebenen, ohne Gelände
fl_ebenen = {"110 kV": fl_flach}
for fl_kv in (220, 380):
    fl_ebenen[f"{fl_kv} kV"], _ = fl_suche(fl_lade_profil(fl_kv))
print(fl_tabelle(fl_ebenen).to_string())
print(FL_SPALTEN)
print("\nMasten je Typ:"); print(fl_typen(fl_ebenen).to_string())

fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
fl_karte(ax, fl_ebenen, ["crimson", "darkorange", "navy"])
fl_legende(ax)
ax.set_title("Drei Spannungsebenen (Punkte = Masten nach Typ)", fontsize=9)
plt.show()

**Was die Zahlen zeigen (ohne Gelände):** Mit höherer Spannung werden die Spannfelder länger (die größte Spannweite folgt `max_span_m`), und die Leitung braucht weniger Masten. Jeder Mast ist aber teurer (größeres Fundament, höherer Grundpreis), deshalb steigen die Gesamtkosten trotzdem stark an. Die Aufhängehöhe der Linie steigt mit dem Profil (25, 35, 40 m); ohne DGM prüft die Suche keinen Bodenabstand und meldet für alle Linienmasten die niedrigste Höhe des Profils.

## 6.4 Mit Gelände (DGM)

Jetzt geben wir das DGM mit `dem=` (als `(Array, CRS, Transform)` auf dem Gitter des Kostenrasters). Damit schalten sich drei Dinge zugleich ein: die Neigungsgrenze `max_gradient_percent` (110 kV: 40 %), der Neigungsfaktor auf Route und Fundament, und die Bodenabstandsprüfung mit Durchhang. Welcher Teil wie viel bewirkt, trennen wir hier nicht.

Im Längsprofil zeichnen wir den Leiter als Parabel, so wie pyorps ihn prüft: `z(x) = zA + (zB - zA) · x / L - w · x · (L - x) / (2 T)` mit den Aufhängepunkten zA, zB (Gelände am Mast plus Aufhängehöhe), Spannweite L, Leitergewicht w und Zugkraft T.

In [ ]:
# 110 kV mit DGM
fl_dgm_erg, _ = fl_suche(fl_p110, dem=FL_DGM)
print("Hinweis von pyorps (englisch), sinngemäß: Die Suche hat nur mit der niedrigsten Masthöhe gesucht und diese "
      "Leitung genommen, ohne sie mit Leitungen mit höheren Masten zu vergleichen:")
print("  search_notes:", fl_dgm_erg.search_notes)
print(fl_tabelle({"ohne DGM": fl_flach, "mit DGM": fl_dgm_erg}).to_string())
print(FL_SPALTEN)

fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
fl_karte(ax, {"ohne DGM": fl_flach, "mit DGM": fl_dgm_erg}, ["crimson", "deepskyblue"], hintergrund="schummerung")
fl_legende(ax)
ax.set_title("110 kV ohne und mit DGM (Schummerung)", fontsize=9)
plt.show()


# Längsprofil: Gelände, Masten und Leiter entlang der Route
def fl_laengsprofil(erg, profil, ax, oberflaeche=None, titel="", legende=True):
    """Zeichnet das Längsprofil; gibt den kleinsten Leiterabstand je mittlerem Spannfeld zurück [m].

    Leiter: Parabel aus pyorps (span_clearance.py): z(x) = zA + (zB - zA) x / L - w x (L - x) / (2 T).
    """
    xy = np.asarray(erg.path_coords)
    lauf = np.r_[0.0, np.cumsum(np.hypot(*np.diff(xy, axis=0).T))]
    zeile, spalte = rasterio.transform.rowcol(FL_T, xy[:, 0], xy[:, 1])
    zeile, spalte = np.asarray(zeile), np.asarray(spalte)
    boden = fl_dgm[zeile, spalte]
    oben = boden if oberflaeche is None else oberflaeche[zeile, spalte]
    ax.fill_between(lauf, boden.min() - 20, boden, color="0.8", label="Gelände (DGM)")
    if oberflaeche is not None:
        ax.plot(lauf, oben, color="darkgreen", lw=0.8, label="angenommene Baumkronen (DOM)")
    # Mastpositionen entlang der Route (jeder Mast steht auf einer Zelle der Route)
    pos, start = [], 0
    for t in erg.towers:
        d = np.hypot(xy[start:, 0] - t.location.x, xy[start:, 1] - t.location.y)
        start = start + int(np.argmax(d < 0.5)); pos.append(start)
    w, T = profil.conductor_weight_per_m, profil.conductor_tension_n
    abstand = []
    for i in range(len(pos) - 1):
        a, b = pos[i], pos[i + 1]
        za, zb = boden[a] + erg.towers[i].height_m, boden[b] + erg.towers[i + 1].height_m
        L = lauf[b] - lauf[a]
        x = lauf[a:b + 1] - lauf[a]
        z = za + (zb - za) * x / L - w * x * (L - x) / (2 * T)
        endfeld = i in (0, len(pos) - 2)
        zu_tief = False
        if not endfeld and b - a > 1:
            abstand.append(float(np.min(z[1:-1] - oben[a + 1:b])))
            zu_tief = abstand[-1] < profil.min_clearance_m
        if zu_tief:   # Spannfeld unter dem Mindestabstand: dick und rot
            ax.plot(lauf[a:b + 1], z, color="red", lw=2.5)
        else:
            ax.plot(lauf[a:b + 1], z, color="tab:blue", lw=1, ls="--" if endfeld else "-")
    for k, t in zip(pos, erg.towers):
        ax.plot([lauf[k], lauf[k]], [boden[k], boden[k] + t.height_m], color=FL_TYPEN[t.tower_type][1], lw=2)
    ax.set_xlabel("Entfernung entlang der Route [m]", fontsize=8); ax.set_ylabel("Höhe [m ü. NHN]", fontsize=8)
    ax.set_title(titel, fontsize=9); ax.tick_params(labelsize=8)
    if legende:   # Platzhalter-Linien nur für die Legende
        ax.plot([], [], color="tab:blue", lw=1, label="Leiter")
        ax.plot([], [], color="tab:blue", lw=1, ls="--", label="Leiter im Endfeld (Höhe des Endmasts nur Bericht)")
        ax.plot([], [], color="red", lw=2.5, label="Leiter zu tief (unter min_clearance_m)")
        ax.plot([], [], color="tab:orange", lw=2, label="Mast bis zur Aufhängehöhe (Farbe = Typ)")
        ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)
    return np.array(abstand)


fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
fl_abstand_dgm = fl_laengsprofil(fl_dgm_erg, fl_p110, ax, titel="110 kV mit DGM: Längsprofil")
plt.show()
print(f"kleinster Abstand Leiter-Boden in den mittleren Spannfeldern: {de(fl_abstand_dgm.min(), 1)} m "
      f"(Vorgabe min_clearance_m = {de(fl_p110.min_clearance_m)} m; nachgerechnet an den Zellen der Route)")

## 6.5 Vertiefung (optional): Bodenabstand über Wald mit einem angenommenen Oberflächenmodell (DOM)

Für den Bodenabstand zählt nicht nur der Boden, sondern alles, was darauf steht: Bäume und Gebäude. Das beschreibt ein **digitales Oberflächenmodell** (DOM, englisch DSM, Argument `dsm=`). pyorps rechnet die Hindernishöhe als `max(DOM - DGM, 0)` und verlangt den Mindestabstand über dem Hindernis.

**Wichtig:** Für unser Gebiet haben wir kein offenes DOM. Wir bauen deshalb ein **angenommenes DOM nur für den Unterricht**: Gelände plus 25 m Baumhöhe auf allen ALKIS-Flächen der Klassen Laubholz, Nadelholz, Laub- und Nadelholz und Gehölz, sonst 0 m. Das ist grob (keine Lichtungen, keine Gebäude, gleiche Baumhöhe überall). Für echte Planung nehmt ihr ein gemessenes Oberflächenmodell aus Laserscanning, zum Beispiel das DOM1 der Landesvermessung. Das DOM liegt auf demselben Gitter wie DGM und Kostenraster, aus dem gleichen Grund wie beim DGM.

**Diese Suche ist die langsamste des Notebooks:** In unserem Lauf dauerte sie etwa 20 bis 30 Sekunden auf einem schnellen Laptop und belegte gut 3 GB Arbeitsspeicher. Mit nur 25-m-Masten findet pyorps hier keine Leitung, deshalb sucht es über alle Höhenklassen (25, 34, 42 m).

In [ ]:
# Angenommenes DOM: Gelände + 25 m auf Waldflächen (nur zum Lernen!)
fl_alkis = gpd.read_file(f"data/layers/alkis_{BBOX[0]}_{BBOX[1]}_{BBOX[2]}_{BBOX[3]}.gpkg")
FL_WALD = ("Laubholz", "Nadelholz", "Laub- und Nadelholz", "Gehölz")
FL_BAUMHOEHE_M = 25.0
fl_baeume = rasterize(((g, FL_BAUMHOEHE_M) for g in fl_alkis.loc[fl_alkis["unterart_name"].isin(FL_WALD), "geometry"]),
                      out_shape=fl_kosten.shape, transform=FL_T, fill=0.0, dtype="float32")
fl_dom = fl_dgm + fl_baeume
print(f"Waldanteil im Raster: {de(100 * (fl_baeume > 0).mean(), 1)} %, "
      f"größte Hindernishöhe DOM - DGM: {de(float((fl_dom - fl_dgm).max()), 1)} m")

fl_dom_erg, _ = fl_suche(fl_p110, dem=FL_DGM, dsm=(fl_dom, FL_CRS, FL_T))
print("search_notes:", fl_dom_erg.search_notes or "leer (die Suche lief über alle Höhenklassen)")
print(fl_tabelle({"mit DGM": fl_dgm_erg, "mit DGM + DOM": fl_dom_erg}).to_string())
print(FL_SPALTEN)
print(f"Höhenaufpreise der Leitung mit DOM: {de(fl_dom_erg.cost_breakdown.get('height_premiums', 0.0))} EUR")

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharey=True, constrained_layout=True)
fl_abstand_alt = fl_laengsprofil(fl_dgm_erg, fl_p110, axes[0], oberflaeche=fl_dom, legende=False,
                                 titel="Leitung nur mit DGM gesucht, gemessen am angenommenen DOM")
fl_abstand_neu = fl_laengsprofil(fl_dom_erg, fl_p110, axes[1], oberflaeche=fl_dom,
                                 titel="Leitung mit DGM + DOM gesucht")
plt.show()
for fl_name, fl_e, fl_a in (("nur DGM", fl_dgm_erg, fl_abstand_alt), ("DGM + DOM", fl_dom_erg, fl_abstand_neu)):
    fl_xy = np.asarray(fl_e.path_coords)
    fl_im_wald = (fl_baeume[rasterio.transform.rowcol(FL_T, fl_xy[:, 0], fl_xy[:, 1])] > 0).mean()
    print(f"{fl_name}: {de(100 * fl_im_wald)} % der Routenzellen im angenommenen Wald; kleinster Abstand "
          f"Leiter-Baumkrone in den mittleren Spannfeldern {de(fl_a.min(), 1)} m; Spannfelder unter "
          f"{de(fl_p110.min_clearance_m)} m: {(fl_a < fl_p110.min_clearance_m).sum()} von {len(fl_a)}")

**Was wir sehen:** Die Leitung, die nur mit dem DGM gesucht wurde, hängt über dem angenommenen Wald an mehreren Stellen zu tief. Mit DOM setzt die Suche dort höhere Masten (34 m und 42 m statt 25 m) und verschiebt einzelne Masten; die Gesamtkosten steigen um die Höhenaufpreise und die geänderten Standorte. Der Effekt ist hier moderat: Nur ein kleiner Teil der Routenzellen liegt im angenommenen Wald (siehe Ausgabe).

Zwei Grenzen der Suche (stehen auch in `search_notes` und in der pyorps-Doku):
- Mit mehreren Masthöhen sucht pyorps zuerst nur mit der niedrigsten Höhe und nimmt diese Leitung, wenn es eine gibt. Ob ein einzelner höherer Mast billiger wäre, prüft es dann nicht. Mit DGM allein ist das hier passiert (alle Linienmasten 25 m).
- Die Suche prüft das letzte Spannfeld nicht und rechnet das erste mit einer Höhe der Linie. Die Höhe des Endmasts im Ergebnis (15 m) ist ein Berichtswert; deshalb sind die Endfelder im Längsprofil gestrichelt.

## 6.6 Vertiefung (optional): Die Leitungsenden, Endfeldregel und Anschlusshöhen

Die Suche hält das erste Spannfeld mindestens `min_span_m` lang, nimmt das Ziel aber in jeder Entfernung an. So kann ein Mast sehr nah am Ziel stehen. Was mit ihm passiert, sagt `terminal_span_rule` im Profil:
- `"remove"` (so steht es in den Profildateien): Der Mast wird nach der Suche gestrichen. Seine Kosten fehlen dann in `total_cost`, und es kann ein Knick ohne Mast bleiben (pyorps warnt mit `SpanRuleWarning`).
- `"remove_if_valid"`: nur streichen, wenn die Leitung gültig bleibt; sonst behalten und in `span_notes` vermerken. **Das haben wir in diesem Teil überall benutzt** (`fl_lade_profil` überschreibt den Wert der Datei).
- `"keep"`: jeden Mast behalten, kurze Endfelder nur vermerken.
- `"enforce"`: die Suche selbst hält jedes Endfeld mindestens `min_span_m` lang. Dafür braucht sie exakte Spannweitenklassen (eine Klasse pro Rasterschritt) und viel mehr Speicher.

**Anschlusshöhen** (`source_connection_height_m`, `target_connection_height_m`, Meter über Gelände): die Höhe, in der der Leiter am Umspannwerk oder an einem vorhandenen Mast hängt. Sie wirken nur mit DGM und Bodenabstandsprüfung; dann prüft die Suche auch das erste und das letzte Spannfeld, ebenfalls mit exakten Spannweitenklassen.

In [ ]:
# a) Dieselbe Suche mit DGM, aber mit der Regel "remove" aus der Profildatei
print("Gleich kommt eine englische SpanRuleWarning. Sie sagt: Der Mast 35 m vor dem Ziel wurde gestrichen, "
      "und dort bleibt ein Knick ohne Mast.")
fl_remove, _ = fl_suche(fl_lade_profil(110, "remove"), dem=FL_DGM)
fl_regeln = pd.DataFrame({
    name: {"Masten": e.n_towers, "Gesamt [EUR]": round(e.total_cost), "gestrichene Masten": len(e.removed_towers),
           "deren Kosten [EUR]": round(e.cost_breakdown.get("removed_towers", 0))}
    for name, e in (("remove", fl_remove), ("remove_if_valid", fl_dgm_erg))}).T
print(fl_regeln.to_string())
print("span_notes bei remove_if_valid (englisch, sinngemäß: kurzes Endfeld behalten, sonst bliebe ein Knick ohne Mast):")
print("  ", fl_dgm_erg.span_notes)

# b) "enforce" im ganzen Raster: exakte Klassen brauchen zu viel Speicher, pyorps lehnt vor der Suche ab
try:
    fl_suche(fl_lade_profil(110, "enforce"), dem=FL_DGM)
except InvalidInputError as fl_fehler:
    print("\nenforce im ganzen Raster abgelehnt, weil die exakte Suche zu viele Zustände bräuchte "
          "(Meldung englisch, die Zahl ist die Anzahl der Zustände):")
    print("  ", str(fl_fehler)[:230], "...")

# c) Kleines Fenster im Talboden (FL_TAL, 1 km nach Süden): ohne und mit Anschlusshöhen 20 m / 12 m
fl_tal = {"ohne Anschlusshöhen": fl_suche(fl_p110, *FL_TAL, puffer=250, dem=FL_DGM)[0],
          "Quelle 20 m, Ziel 12 m": fl_suche(fl_p110, *FL_TAL, puffer=250, dem=FL_DGM,
                                             source_connection_height_m=20.0, target_connection_height_m=12.0)[0]}
print(pd.DataFrame({
    name: {"Masten": e.n_towers, "Gesamt [EUR]": round(e.total_cost),
           "Endmast Quelle / Ziel [m]": f"{e.towers[0].height_m:g} / {e.towers[-1].height_m:g}",
           "erste / letzte Spannweite [m]": f"{e.spans[0]:.0f} / {e.spans[-1]:.0f}",
           "Spannweitenklasse [m]": de(e.span_bin_size_m, 2)}
    for name, e in fl_tal.items()}).T.to_string())
print("search_notes (englisch, sinngemäß: exakte Spannweitenklassen von knapp 5 m statt 50 m):")
print("  ", fl_tal["Quelle 20 m, Ziel 12 m"].search_notes)

**Was wir sehen:** Mit `"remove"` sieht die Leitung billiger aus, aber nur weil der gestrichene Mast nicht mehr in `total_cost` zählt; dafür bleibt kurz vor dem Ziel ein Knick ohne Mast. `"remove_if_valid"` behält den Mast und sagt es in `span_notes`. Im Talboden ändern die Anschlusshöhen die Kosten nicht; die exakte Suche setzt aber den letzten Linienmast an eine andere Stelle, und die Endmasten tragen die angegebenen Höhen. Im ganzen Raster sind exakte Klassen zu groß: Dafür brauchen wir ein kleineres Fenster, eine kürzere `max_span_m` oder ein gröberes Raster.

## 6.7 Ergebnisse speichern

Route und Masten gehen als GeoPackage (zwei Layer) und die Masten zusätzlich als GeoJSON nach `data/ergebnisse/`. Wir speichern die Leitung mit DOM, wenn ihr die Vertiefung 6.5 gerechnet habt, sonst die Leitung mit DGM aus 6.4; die Zelle sagt, welche. Die Masten sind Quadrate in der Größe der Fundamentfläche (10 m × 10 m), gedreht in die Winkelhalbierende der Leitung; in QGIS könnt ihr sie direkt anzeigen.

In [ ]:
# Export: die Leitung mit DOM (6.5), sonst mit DGM (6.4), sonst ohne Gelände (6.2)
for fl_export_name, fl_export_var in (("mit DGM + DOM (6.5)", "fl_dom_erg"), ("mit DGM (6.4)", "fl_dgm_erg"),
                                      ("ohne Gelände (6.2)", "fl_flach")):
    if fl_export_var in globals():
        fl_export = globals()[fl_export_var]
        break
else:
    raise NameError("Noch keine Leitung gerechnet: bitte zuerst die Zellen ab 6.2 ausführen.")
print("Wir speichern die 110-kV-Leitung", fl_export_name)
FL_ERGEBNISSE = Path("data/ergebnisse")
FL_ERGEBNISSE.mkdir(parents=True, exist_ok=True)
fl_gpkg = FL_ERGEBNISSE / "freileitung_110kv.gpkg"
fl_gpkg.unlink(missing_ok=True)                 # bei erneutem Lauf neu schreiben

fl_route = gpd.GeoDataFrame(
    {"profil": [fl_export.profile_name], "variante": [fl_export_name], "masten": [fl_export.n_towers],
     "laenge_m": [fl_export.total_length], "gesamt_eur": [fl_export.total_cost],
     "endfeldregel": [fl_export.terminal_span_rule]},
    geometry=[fl_export.path_geometry], crs=FL_CRS)
fl_mastflaechen = fl_export.towers_to_geodataframe(crs=FL_CRS)
fl_route.to_file(fl_gpkg, layer="route", driver="GPKG")
fl_mastflaechen.to_file(fl_gpkg, layer="masten", driver="GPKG")
fl_mastflaechen.to_file(FL_ERGEBNISSE / "masten_110kv.geojson", driver="GeoJSON")

print(gpd.list_layers(fl_gpkg).to_string(index=False))
print(f"Masten zurückgelesen: {len(gpd.read_file(fl_gpkg, layer='masten'))} | Spalten: {list(fl_mastflaechen.columns)}")

## Was wir mitnehmen

- `ConstrainedPathFinder` sucht Route **und** Masten zusammen; das Profil (`InfrastructureProfile`, YAML-Datei) enthält alle Regeln.
- Die Masten dominieren die Kosten. Höhere Spannung heißt hier: längere Spannfelder, weniger, aber teurere Masten.
- Verbotene Zellen sperren auch den Leiter. Enge Lücken im Kostenraster zwingen eine Freileitung zu Umwegen; ein zu kleines Suchfenster gibt ein **leeres Ergebnis ohne Warnung**.
- DGM und DOM für `ConstrainedPathFinder` immer auf dem Gitter des Kostenrasters übergeben.
- Mit DGM werden Neigung und Bodenabstand geprüft; mit DOM auch Bäume. Das DOM in diesem Notebook ist eine Annahme zum Lernen.
- `search_notes`, `span_notes` und die Endfeldregel lesen: Sie sagen, wo die Suche abkürzt.

**In der GUI** (sie ist englisch beschriftet): Der Schalter **"Constrained routing (overhead line / towers)"** im Routing-Bereich blendet die Profiloptionen ein; "Run routing" rechnet dann eine Freileitung auf denselben Punkten (jeder Zwischenpunkt erzwingt einen Mast). Das Profil erscheint als Formular mit 24 Feldern und drei Tabellen:

| GUI-Feld | Profilfeld |
|---|---|
| Soft turn limit / Hard turn limit (forbidden above) | `soft_angle_limit_deg` / `hard_angle_limit_deg` |
| Turn cost shape, Turn cost factor, Tabelle der Knickkosten | `angle_cost_function`, `angle_cost_params` |
| Shortest span / Longest span / Span step | `min_span_m` / `max_span_m` / `span_bin_size_m` |
| Tabelle der Fundamentkosten, Between table values | `terrain_cost_map` (EUR pro m²), `terrain_interpolation` |
| Tabelle der Masttypen | `angle_types` |
| Steepest allowed slope / Slope penalty factor | `max_gradient_percent` / `gradient_cost_params["scale"]` |
| Base attachment height / Height options / Extra cost per 3 m step | `tower_height_m` / `tower_heights_m` / `tower_height_cost_per_increment` |
| Height at the end points / Cost of each end tower | `terminal_tower_height_m` / `terminal_tower_cost` |
| Conductor weight / Conductor tension / Ground clearance | `conductor_weight_per_m` / `conductor_tension_n` / `min_clearance_m` |
| Foundation area / Foundation pricing | `tower_ground_area_m2` / `tower_area_cost_mode` |
| End-span rule | `terminal_span_rule` |

Dazu kommen Name, Beschreibung und "Suggested corridor width (info only)". Die Anschlusshöhen stehen im Block "Line ends".

### Übung Teil 6

1. **Feinere Richtungen:** Rechnet die Leitung ohne Gelände mit 32 Richtungen: `fl_suche(fl_p110, nachbarschaft="r3")`. Wie ändern sich Zahl der Masten, Gesamtkosten und Rechenzeit? **Achtung:** `r3` im ganzen Raster braucht mehr Arbeitsspeicher als `r2` und kann einen Laptop mit 8 GB an die Grenze bringen. Mit wenig Speicher nehmt ihr das kleine Talpaar: `fl_suche(fl_p110, *FL_TAL, puffer=250, nachbarschaft="r3")` und vergleicht mit `nachbarschaft="r2"`.
2. **Spannungsebenen mit Gelände:** Sucht die 380-kV-Leitung mit `dem=FL_DGM` und vergleicht sie mit `fl_dgm_erg` (110 kV). Gilt "höhere Spannung, weniger Masten" auch im Gelände? Schaut auf `max_gradient_percent` und `hard_angle_limit_deg` der beiden Profile.
3. **Kürzere Spannfelder:** Sucht mit dem Profil `fl_kurz` (`max_span_m = 200`) ohne Gelände. Wie viele Masten kommen dazu, und was kostet das?

---
# Teil 7: Die GUI lokal betreiben

Alles, was wir im Code gemacht haben, geht auch **ohne Code** in der grafischen Oberfläche von PYORPS
(`pyorps.gui`, eine Dash-Anwendung mit Karte). Sie läuft **auf eurem eigenen Rechner**: Ein kleiner Webserver
startet lokal, und ihr bedient ihn im Browser. Eure Daten verlassen den Rechner nicht.

## 7.1 Starten

Im Terminal, in der Umgebung, in der PYORPS installiert ist (Teil 1):

```bash
python -m pyorps.gui                      # öffnet die Adresse im Browser (Standard: http://127.0.0.1:8050)
python -m pyorps.gui --desktop            # eigenes Fenster (pywebview) statt Browser-Tab
python -m pyorps.gui --port 8060          # anderer Port; ist er belegt, nimmt PYORPS den nächsten freien
python -m pyorps.gui --work-dir gui_arbeit             # Ordner für Raster und temporäre Dateien
python -m pyorps.gui --project mein_gebiet/project.json  # gespeichertes Projekt direkt öffnen
```

| Option | Wirkung |
|---|---|
| `--desktop` | Fenster statt Browser (braucht `pywebview`, im Zusatz `[gui]` enthalten; unter Linux kann `pywebview` zusätzlich GTK oder Qt verlangen, dann hilft der Browser) |
| `--host`, `--port` | Adresse und Port; Standard `127.0.0.1` und `8050` |
| `--allow-remote` | erlaubt eine andere Adresse als `127.0.0.1` |
| `--work-dir` | Arbeitsordner; wird nie automatisch gelöscht |
| `--project DATEI` | öffnet ein gespeichertes Projekt (`project.json` oder der Ordner darin) |
| `--threads N` | Threads des Webservers (Standard 8) |

> **Sicherheit:** Die GUI hat **keine Anmeldung**, und ihre Pfadfelder erreichen jede Datei des Benutzers.
> Deshalb hört sie nur auf `127.0.0.1`. `--allow-remote` nur in einem vertrauenswürdigen Netz verwenden, nie im Internet.
> Für einen gemeinsamen Rechner im Seminarraum startet jede Person ihre **eigene** GUI auf dem eigenen Laptop.

## 7.2 Aus dem Notebook starten (optional)

Das Starten ist **nicht** Teil des normalen Laufs: Der Schalter `GUI_STARTEN` steht auf `False`. Setzt ihn auf `True`,
startet die nächste Zelle die GUI als Hintergrundprozess und schreibt die Meldungen in `data/gui_log.txt`.
Die Zelle danach beendet sie wieder. Ein Notebook-Lauf von oben nach unten startet also nie ungewollt einen Server.

In [ ]:
import subprocess
import sys
import time
from pathlib import Path

GUI_STARTEN = False      # auf True setzen, um die GUI aus dem Notebook zu starten
gui_prozess = None

if GUI_STARTEN:
    arbeitsordner = Path("data/gui_arbeit").resolve()
    arbeitsordner.mkdir(parents=True, exist_ok=True)
    log = open("data/gui_log.txt", "w", encoding="utf-8")
    gui_prozess = subprocess.Popen(
        [sys.executable, "-m", "pyorps.gui", "--port", "8050", "--work-dir", str(arbeitsordner)],
        stdout=log, stderr=subprocess.STDOUT,
    )
    time.sleep(8)
    print("GUI läuft (Prozess", gui_prozess.pid, ") – im Browser öffnen: http://127.0.0.1:8050")
    print("(Ist der Port belegt, nimmt PYORPS den nächsten freien: die richtige Adresse steht in der Log-Datei.)")
    print("Meldungen:", Path("data/gui_log.txt").resolve())
else:
    print("GUI_STARTEN ist False – es wurde nichts gestartet.")

In [ ]:
# GUI beenden (nur nötig, wenn sie oben gestartet wurde)
if gui_prozess is not None:
    gui_prozess.terminate()
    gui_prozess.wait(timeout=15)
    print("GUI beendet.")
else:
    print("Keine GUI aus dem Notebook gestartet.")

## 7.3 Notebook und GUI im Vergleich

Die GUI ist englisch beschriftet und hat sechs Reiter. Jeder entspricht einem Schritt dieses Workshops:

| Reiter in der GUI | Was dort passiert | Entsprechung im Notebook |
|---|---|---|
| **Data** | Gebiet auf der Karte zeichnen; Daten laden (lokale Datei, WFS, OpenStreetMap, Geländemodell) | Teil 3: `lade_layer`, WFS-Abfragen; Teil 5: DGM |
| **Cost** | Kostentabelle je Layer erzeugen (*Seed*) und bearbeiten | Teil 3: `CostAssumptions` |
| **Raster** | Vektordaten in ein Kostenraster umwandeln (*Build cost raster*) | Teil 3: `GeoRasterizer` |
| **Routes** | Quelle und Ziel setzen, Algorithmus und Nachbarschaft wählen, Routen berechnen (*Run routing*) und bearbeiten | Teil 4: `PathFinder.find_route` |
| **Layers** | Tabelle der Kartenebenen: Namen ändern, ein- und ausblenden | – |
| **Attrs** | Ein Objekt auf der Karte anklicken, seine Attribute erscheinen hier | Blick in die Ergebnistabellen |

Die GUI führt euch durch fünf Schritte und hakt sie ab, sobald sie erledigt sind:
**Gebiet zeichnen → Daten laden → Kosten festlegen → Kostenraster bauen → Routen planen.**

Im Reiter **Routes** liegen eingeklappte Karten für die Themen aus den Teilen 5 und 6:

- **Objective & terrain:** Gewichte für Kosten, Länge und Steigung, Geländemodell (DGM) – wie in Teil 5.
  *Die GUI lehnt Geländemodell zusammen mit Delta-Stepping ab* und weist auf das Feld „Algorithm“ hin; die Bibliothek
  fällt dagegen mit einer Warnung auf Dijkstra zurück (Teil 5).
- **Constrained routing (overhead line / towers):** Profil als Formular (Winkel, Spannfeld, Mastkosten), wie in Teil 6.
- **Cost field:** die Kosten von den Quellen zu jeder Zelle als Rasterebene – das Ergebnis der Mehr-Quellen-Rechnung aus Teil 4.
- **Corridor graph:** gemeinsame Gräben für mehrere Routen.

Mit **Save project** speichert die GUI Gebiet, Kostentabelle, Raster, Routen und alle Einstellungen in einen Ordner
(darin liegt die `project.json`); mit **Open project** oder `--project` öffnet ihr es wieder.

### Übung Teil 7

1. Startet die GUI im Terminal (`python -m pyorps.gui`) und zeichnet ein kleines Gebiet (etwa 2 × 2 km) in dem Gebiet
   aus Teil 3 (Kinzigtal bei Biberach im Ortenaukreis, Baden-Württemberg).
2. Ladet die ALKIS-Landnutzung über WFS (die Adresse steht in Teil 3), prüft die Kostentabelle im Reiter **Cost**,
   baut das Raster und berechnet eine Route. Vergleicht sie mit der Route aus dem Notebook.
3. Speichert das Projekt, beendet die GUI und öffnet es mit `--project` wieder.

---
# Teil 8: Zusammenfassung

## Was wir gemacht haben

| Teil | Kernaussage |
|---|---|
| 1 Installation | `pip install "pyorps[gui,examples]"` in einer eigenen Umgebung; keine Compiler nötig |
| 2 Grundidee | Raster → Graph → kürzeste Route; niedriger Kostenwert = billig, 65535 = gesperrt |
| 3 Kostenraster | Ein Basislayer (Landnutzung) plus überlagernde Layer (sperren, Zuschläge, Abschläge); andere Annahmen ergeben andere Trassen |
| 4 Algorithmen | Dijkstra und Delta-Stepping liefern dieselben Kosten; Delta-Stepping arbeitet Eimer parallel ab |
| 5 Gelände | Mit Geländemodell zählen Steigung und 3D-Länge; Delta-Stepping unterstützt das nicht (Rückfall auf Dijkstra) |
| 6 Freileitung | `ConstrainedPathFinder` sucht Trasse und Maststandorte gemeinsam, mit Winkel- und Spannfeldgrenzen |
| 7 GUI | Dieselbe Planung im Browser, lokal auf dem eigenen Rechner |

## Welches Verfahren wofür?

| Aufgabe | Werkzeug |
|---|---|
| Ein Erdkabel von A nach B | `PathFinder`, Nachbarschaft `r2`; `algorithm="dijkstra"` oder `"delta-stepping"` (gleiche Kosten, Zeiten selbst messen) |
| Viele Ziele oder Quellen, großes Raster | `PathFinder` oder Kostenfeld, beide Algorithmen vergleichen (Teil 4) |
| Hügeliges Gelände, Steigungsgrenze | `PathFinder` mit Geländemodell und `Objective` (Dijkstra) |
| Freileitung mit Masten und Spannfeldern | `ConstrainedPathFinder` mit `InfrastructureProfile` |
| Planung ohne Code, mit Karte | `python -m pyorps.gui` |

## Weiterführend

- **Dokumentation:** [pyorps.readthedocs.io](https://pyorps.readthedocs.io), darin *Release Status*, *Neighborhoods*,
  *Cost semantics* und die API-Seiten zu `PathFinder`, `ConstrainedPathFinder` und der GUI
- **Quellcode und Fehlermeldungen:** [github.com/marhofmann/pyorps](https://github.com/marhofmann/pyorps)
- **Weitere Funktionen, die wir nicht behandelt haben:** GPU-Rechnung (`pip install "pyorps[gpu]"`, Backends
  `raster_gpu` und `raster_fim`), Kostenfelder und Suchsitzungen für viele Endpunktpaare, Korridorgraphen für gemeinsame
  Gräben, vorberechnete Mastfelder
- **Zitieren:** Hofmann, M., Stetz, T., Kammer, F., Repo, S.: „PYORPS: An Open-Source Tool for Automated Power Line Routing“,
  CIRED 2025 – 28th Conference and Exhibition on Electricity Distribution, 16.–19. Juni 2025, Genf

**Kontakt:** Martin Hofmann, THM – Technische Hochschule Mittelhessen, Gießen

---
*Datenquellen dieses Workshops: © LGL-BW (ALKIS, Tatsächliche Nutzung; DGM1) und © LUBW (Schutzgebiete, Biotope,
Überschwemmungsgebiete), Datenlizenz Deutschland – Namensnennung – Version 2.0 (www.govdata.de/dl-de/by-2-0). Die Daten
wurden zugeschnitten, gerastert, gemittelt und umklassifiziert. Das Geländemodell steht mit seiner Quelle in Teil 5.*